In [1]:
from pyspark.sql import SparkSession

jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]

fraud_channels =  ['API', 'ATM', 'BIO', 'BISP', 'Business App', 'Business App API', 'Cheetay', 'JC Keyboard', 'Merchant Payment', 'Mobile App', 'NEW_JC_APP', 'PAYPAK', 'PGW', 'Payment Gateway', 'QR Payment', 'Self Care App', 'THIRD_PARTY_WEB', 'USSD', 'USSD API', 'USSD_API', 'VRG']
fraud_types =  ['Online Payment', 'PTS ATM Withdrawal', 'PTS Purchase Payment', 'Transfer(C2C)', 'MFS Card Withdraw', 'Cash in', 'Cash out', 'IBFT Outgoing Customer', 'IBFT Outgoing OTC', 'Merchant Payment', 'Transfer(B2C)', 'Transfer(C2B)', 'Jazz Load (Prepaid top-up)', 'Business Cash Out', 'Customer Remit To CNIC', 'Donation', 'Get Loan', 'IBFT Credit', 'Others', 'Utility Bills Payment', 'Purchase Payment', 'Auto Debit', 'Indigo Bills (Postpaid payment)', '']

channels_in_clause = ", ".join([f"'{ch}'" for ch in fraud_channels])
types_in_clause = ", ".join([f"'{tp}'" for tp in fraud_types])
# selected_cols = ['trans_id', 'ac_from', 'ac_to', 'data_date', 'trans_initiate_time', 'cutoff_date', 'fraud_flag', 'trx_channel', 'trx_type', 'start_balance', 'end_balance', 'trx_amt', 'mbar_registered_date_time', 'mbar_a_c_status',  'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'is_business_hours', 'is_unusual_hour', 'night_weekend_combo', 'start_balance_log', 'balance_change', 'balance_change_pct', 'txn_txns_3d', 'txn_total_amount_3d', 'txn_avg_amount_3d', 'txn_max_amount_3d', 'txn_min_amount_3d', 'txn_unique_recipients_3d', 'txn_unique_channels_3d', 'txn_unique_types_3d', 'txn_is_high_activity_3d', 'txn_multi_channel_recent', 'txn_amount_deviation_from_avg', 'txn_night_txns_3d', 'txn_weekend_txns_3d', 'channel_new_jc_app', 'channel_ussd', 'channel_ussd_api', 'channel_payment_gateway', 'channel_mobile_app', 'type_transfer_c2c', 'type_transfer_c2b', 'type_bill_payment', 'type_mobile_load', 'user_total_txns_3d', 'user_total_amount_3d', 'user_avg_amount_3d', 'user_median_amount_3d', 'user_max_amount_3d', 'user_min_amount_3d', 'user_unique_recipients_3d', 'user_unique_channels_3d', 'user_unique_types_3d', 'user_total_txns_7d', 'user_total_amount_7d', 'user_avg_amount_7d', 'user_median_amount_7d', 'user_max_amount_7d', 'user_min_amount_7d', 'user_unique_recipients_7d', 'user_unique_channels_7d', 'user_unique_types_7d', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_channel_diversity_score_7d', 'user_most_used_type_7d', 'user_last_used_type', 'user_type_diversity_score_7d', 'user_night_txns_7d', 'user_weekend_txns_7d', 'user_peak_hour_txns_7d', 'user_off_peak_hour_txns_7d', 'user_avg_start_balance_7d', 'user_avg_end_balance_7d', 'user_min_balance_7d', 'user_max_balance_7d', 'user_balance_volatility_7d', 'user_top_recipient_7d', 'user_avg_amount_per_recipient_7d', 'user_max_amount_to_single_recipient_7d', 'user_recipient_concentration_ratio_7d', 'user_avg_time_between_txns_7d', 'user_txn_frequency_score_7d', 'user_first_txn_time', 'user_last_txn_time', 'user_days_since_last_txn']

CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

# spark.stop()
# Initialize Spark session with JARs
spark = SparkSession.builder \
    .appName("data_loading") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "150g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()

# CORRECT URL: Use HTTP port 8123 (not native port 9000)
url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user'] 
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

25/11/11 17:21:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [8]:

start_date = '2025-05-01'
end_date = '2025-06-30'
query = f"""
    SELECT *
    FROM stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
    AND mbar_account_type_name = 'Customer Account'
"""
subquery = f"""
(
    {query}
) AS fraud_data
"""

df = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', subquery)
    .option('fetchsize', '1000000')  # Fetch 100k rows at a time
    .option("partitionColumn", "cutoff_date") \
    .option('lowerBound', start_date)  # Lower bound of partition column
    .option('upperBound', end_date)    # Upper bound of partition column
    .option('numPartitions', str(120))  # Number of partitions
    .load())

25/11/11 15:41:09 WARN JDBCRelation: The number of partitions is reduced because the specified number of partitions is less than the difference between upper bound and lower bound. Updated number of partitions: 60; Input number of partitions: 120; Lower bound: '2025-05-01'; Upper bound: '2025-06-30'.


In [9]:
df.count()

377490327

In [10]:
df.groupBy('trx_channel').sum('fraud_flag').show()

+--------------------+---------------+
|         trx_channel|sum(fraud_flag)|
+--------------------+---------------+
|       Self Care App|            779|
|                 API|             71|
|     THIRD_PARTY_WEB|           3441|
|                BISP|            107|
|Insurance Subscri...|              0|
|                VISA|              0|
|                 ATM|              2|
|                 CMS|              0|
|                USSD|            389|
|                 VRG|              2|
|    Business App API|              1|
|             Cheetay|            100|
|                  IR|              0|
|          Mobile App|             11|
|     Payment Gateway|          10000|
|     CorporatePortal|              0|
|          QR Payment|            274|
|                 BIO|              0|
|                 PGW|              1|
|                 IVR|              0|
+--------------------+---------------+
only showing top 20 rows


In [11]:
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round

# Calculate channel-wise fraud statistics
channel_stats = df.groupBy('trx_channel').agg(
    count('*').alias('total_transactions'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_sum(col('fraud_flag') * col('trx_amt')).alias('fraud_amount'),
    spark_sum('trx_amt').alias('total_amount')
)

# Calculate fraud percentage
channel_stats = channel_stats.withColumn(
    'fraud_percentage',
    spark_round((col('fraud_count') / col('total_transactions')) * 100, 2)
).withColumn(
    'fraud_amount_percentage',
    spark_round((col('fraud_amount') / col('total_amount')) * 100, 2)
).orderBy(col('fraud_count').desc())

# Show results
print("Channel-wise Fraud Analysis:")
channel_stats.show(50, truncate=False)

Channel-wise Fraud Analysis:


+----------------------------+------------------+-----------+--------------------+---------------------+----------------+-----------------------+
|trx_channel                 |total_transactions|fraud_count|fraud_amount        |total_amount         |fraud_percentage|fraud_amount_percentage|
+----------------------------+------------------+-----------+--------------------+---------------------+----------------+-----------------------+
|Payment Gateway             |123288653         |10000      |1.0647666932000001E8|1.3757672830836145E11|0.01            |0.08                   |
|NEW_JC_APP                  |411103649         |9225       |1.21148082E8        |1.8669683276755186E12|0.0             |0.01                   |
|THIRD_PARTY_WEB             |34979219          |3441       |1.6839775E7         |2.4593668065339047E11|0.01            |0.01                   |
|Self Care App               |1155229           |779        |3385930.0           |8.41876442E8         |0.07            |0.4

In [12]:
# Convert to Pandas and save to CSV
import os
from datetime import datetime

pandas_df = channel_stats.toPandas()

# Create output filename with timestamp
output_filename = f"channel_fraud_analysis_{start_date}_to_{end_date}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
output_path = os.path.join('/root/research-dir/dev/jazzcash-fraud-detection/data', output_filename)

# Save to CSV
pandas_df.to_csv(output_path, index=False)

print(f"\n✅ Results saved to: {output_path}")
print(f"📊 Total channels analyzed: {len(pandas_df)}")
print(f"🚨 Total fraud transactions: {pandas_df['fraud_count'].sum():,.0f}")
print(f"📈 Total transactions: {pandas_df['total_transactions'].sum():,.0f}")
print(f"💰 Total fraud amount: ${pandas_df['fraud_amount'].sum():,.2f}")

# Display top 10 channels by fraud count
print("\n🔝 Top 10 Channels by Fraud Count:")
print(pandas_df.head(10).to_string(index=False))


✅ Results saved to: /root/research-dir/dev/jazzcash-fraud-detection/data/channel_fraud_analysis_2025-03-01_to_2025-06-30_20251110_144421.csv
📊 Total channels analyzed: 38
🚨 Total fraud transactions: 25,179
📈 Total transactions: 1,153,913,355
💰 Total fraud amount: $270,392,612.77

🔝 Top 10 Channels by Fraud Count:
    trx_channel  total_transactions  fraud_count  fraud_amount  total_amount  fraud_percentage  fraud_amount_percentage
Payment Gateway           123288653        10000  106476669.32  1.375767e+11              0.01                     0.08
     NEW_JC_APP           411103649         9225  121148082.00  1.866968e+12              0.00                     0.01
THIRD_PARTY_WEB            34979219         3441   16839775.00  2.459367e+11              0.01                     0.01
  Self Care App             1155229          779    3385930.00  8.418764e+08              0.07                     0.40
       USSD_API            69172497          721    7157892.00  3.730562e+11        

In [14]:
# Calculate transaction type-wise fraud statistics
type_stats = df.groupBy('trx_type').agg(
    count('*').alias('total_transactions'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_sum(col('fraud_flag') * col('trx_amt')).alias('fraud_amount'),
    spark_sum('trx_amt').alias('total_amount')
)

# Calculate fraud percentage
type_stats = type_stats.withColumn(
    'fraud_percentage',
    spark_round((col('fraud_count') / col('total_transactions')) * 100, 2)
).withColumn(
    'fraud_amount_percentage',
    spark_round((col('fraud_amount') / col('total_amount')) * 100, 2)
).orderBy(col('fraud_count').desc())

# Show results
print("Transaction Type-wise Fraud Analysis:")
type_stats.show(50, truncate=False)

Transaction Type-wise Fraud Analysis:


+-------------------------------------------+------------------+-----------+--------------------+---------------------+----------------+-----------------------+
|trx_type                                   |total_transactions|fraud_count|fraud_amount        |total_amount         |fraud_percentage|fraud_amount_percentage|
+-------------------------------------------+------------------+-----------+--------------------+---------------------+----------------+-----------------------+
|Online Payment                             |119588355         |9981       |1.0642128432000001E8|1.3593531543474176E11|0.01            |0.08                   |
|Transfer(C2C)                              |205488060         |7437       |9.4209018E7         |9.178822463309496E11 |0.0             |0.01                   |
|Get Loan                                   |17820443          |3440       |1.68124E7           |7.30130877261E10     |0.02            |0.02                   |
|Transfer(C2B)                    

In [15]:
# Convert to Pandas and save to CSV
pandas_type_df = type_stats.toPandas()

# Create output filename with timestamp
type_output_filename = f"type_fraud_analysis_{start_date}_to_{end_date}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
type_output_path = os.path.join('/root/research-dir/dev/jazzcash-fraud-detection/data', type_output_filename)

# Save to CSV
pandas_type_df.to_csv(type_output_path, index=False)

print(f"\n✅ Results saved to: {type_output_path}")
print(f"📊 Total transaction types analyzed: {len(pandas_type_df)}")
print(f"🚨 Total fraud transactions: {pandas_type_df['fraud_count'].sum():,.0f}")
print(f"📈 Total transactions: {pandas_type_df['total_transactions'].sum():,.0f}")
print(f"💰 Total fraud amount: ${pandas_type_df['fraud_amount'].sum():,.2f}")

# Display top 10 transaction types by fraud count
print("\n🔝 Top 10 Transaction Types by Fraud Count:")
print(pandas_type_df.head(10).to_string(index=False))


✅ Results saved to: /root/research-dir/dev/jazzcash-fraud-detection/data/type_fraud_analysis_2025-03-01_to_2025-06-30_20251110_145539.csv
📊 Total transaction types analyzed: 47
🚨 Total fraud transactions: 25,179
📈 Total transactions: 1,153,913,355
💰 Total fraud amount: $270,392,612.77

🔝 Top 10 Transaction Types by Fraud Count:
                  trx_type  total_transactions  fraud_count  fraud_amount  total_amount  fraud_percentage  fraud_amount_percentage
            Online Payment           119588355         9981  106421284.32  1.359353e+11              0.01                     0.08
             Transfer(C2C)           205488060         7437   94209018.00  9.178822e+11              0.00                     0.01
                  Get Loan            17820443         3440   16812400.00  7.301309e+10              0.02                     0.02
             Transfer(C2B)           131885834         1648   18546039.00  6.992752e+11              0.00                     0.00
     Utility B

In [25]:
from pyspark.sql.functions import col

# Create fraud and non-fraud dataframes using fraud_flag filter
fraud_df = df.filter(col('fraud_flag') == 1)
non_fraud_df = df.filter(col('fraud_flag') == 0)

print(f"Total transactions: {df.count():,}")
print(f"Fraud transactions: {fraud_df.count():,}")
print(f"Non-fraud transactions: {non_fraud_df.count():,}")

Total transactions: 377,490,327


Fraud transactions: 13,753


Non-fraud transactions: 377,476,574


In [26]:
# Take 10% sample of non-fraud transactions
non_fraud_sample = non_fraud_df.sample(fraction=0.01, seed=42)

print(f"Original non-fraud transactions: {non_fraud_df.count():,}")
print(f"Sampled non-fraud transactions (10%): {non_fraud_sample.count():,}")

Original non-fraud transactions: 377,476,574


Sampled non-fraud transactions (10%): 3,777,185


In [27]:
# Combine fraud and sampled non-fraud data
balanced_df = fraud_df.union(non_fraud_sample)

print(f"Combined balanced dataset: {balanced_df.count():,}")
print(f"Fraud transactions: {fraud_df.count():,}")
print(f"Non-fraud sample: {non_fraud_sample.count():,}")

# Unpersist other dataframes to free up memory
df.unpersist()
fraud_df.unpersist()
non_fraud_df.unpersist()
non_fraud_sample.unpersist()

print("✅ Unpersisted original dataframes to free up memory")

Combined balanced dataset: 3,790,938
Fraud transactions: 13,753


Non-fraud sample: 3,777,185
✅ Unpersisted original dataframes to free up memory


## Comprehensive Fraud Detection Rules Engine

This section implements 9 categories of fraud detection rules:
1. **Velocity-Based Rules** - Transaction frequency and amount patterns
2. **Time-Based Patterns** - Temporal anomaly detection
3. **Account Age Rules** - New account and dormancy patterns
4. **Amount-Based Rules** - Threshold and pattern detection
5. **Channel-Based Rules** - Multi-channel fraud patterns
6. **Account Type Rules** - Cross-type anomalies
7. **Behavioral Anomaly Rules** - Deviation from normal patterns
8. **Recipient/Beneficiary Patterns** - Money mule detection
9. **Fraud Ring Detection** - Network-based fraud detection

In [28]:
# Import required functions
from pyspark.sql.functions import (
    col, when, lit, count, sum as spark_sum, avg, stddev, min as spark_min, 
    max as spark_max, datediff, unix_timestamp, from_unixtime, window, 
    collect_set, size, countDistinct, lag, lead, abs as spark_abs,
    round as spark_round, floor, expr, coalesce
)
from pyspark.sql.window import Window
from datetime import datetime, timedelta

print("✅ Imported required PySpark functions")
print("🎯 Ready to implement fraud detection rules")

✅ Imported required PySpark functions
🎯 Ready to implement fraud detection rules


In [29]:
# RULE CATEGORY 1: VELOCITY-BASED RULES
print("🚀 Implementing Velocity-Based Rules...")

# Assume we have a dataframe 'df' with transactions
# Let's use the combined_df from previous cells or create window specs

# Define time windows for velocity calculations
velocity_df = balanced_df.withColumn('trans_timestamp', 
    unix_timestamp(col('trans_initiate_time')))

# Window specs for velocity calculations
# 15 minutes = 900 seconds, 1 hour = 3600, 24 hours = 86400, 7 days = 604800

# Rule 1.1: Transaction Count Velocity per Account (sender)
print("   • Rule 1.1: Transaction count velocity...")

window_15min = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-900, 0)
window_1hour = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-3600, 0)
window_24hour = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-86400, 0)
window_7days = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-604800, 0)

velocity_df = velocity_df.withColumn('txn_count_15min', count('*').over(window_15min)) \
    .withColumn('txn_count_1hour', count('*').over(window_1hour)) \
    .withColumn('txn_count_24hour', count('*').over(window_24hour)) \
    .withColumn('txn_count_7days', count('*').over(window_7days))

# Velocity flags
velocity_df = velocity_df.withColumn('rule_velocity_count_15min', 
    when(col('txn_count_15min') > 3, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_1hour', 
    when(col('txn_count_1hour') > 5, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_24hour', 
    when(col('txn_count_24hour') > 10, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_7days_new_account', 
    when((col('txn_count_7days') > 20) & (datediff(col('cutoff_date'), col('mbar_registered_date_time')) < 30), 1).otherwise(0))

print("   ✓ Transaction count velocity rules applied")

# Rule 1.2: Amount Velocity
print("   • Rule 1.2: Amount velocity...")

velocity_df = velocity_df.withColumn('amount_sum_1hour', 
    spark_sum('trx_amt').over(window_1hour)) \
    .withColumn('amount_sum_15min', 
    spark_sum('trx_amt').over(window_15min))

# Calculate daily average for each account (need historical window)
window_account_history = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-30, -1)
velocity_df = velocity_df.withColumn('historical_daily_avg', 
    coalesce(avg('trx_amt').over(window_account_history), lit(0)))

velocity_df = velocity_df.withColumn('rule_velocity_amount_3x_daily', 
    when((col('amount_sum_1hour') > col('historical_daily_avg') * 3) & (col('historical_daily_avg') > 0), 1).otherwise(0)) \
    .withColumn('rule_velocity_amount_15min_threshold', 
    when((col('amount_sum_15min') > 10000) & (col('historical_daily_avg') < 2000), 1).otherwise(0))

print("   ✓ Amount velocity rules applied")

# Rule 1.3: Cross-Feature Velocity
print("   • Rule 1.3: Cross-feature velocity...")

# Multiple channels in short time
window_30min_channels = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-1800, 0)
velocity_df = velocity_df.withColumn('channels_used_30min', 
    size(collect_set('trx_channel').over(window_30min_channels)))

velocity_df = velocity_df.withColumn('rule_velocity_multiple_channels', 
    when(col('channels_used_30min') > 1, 1).otherwise(0))

# Same recipient from multiple senders (calculated per recipient)
# Use collect_set instead of countDistinct for window functions
window_recipient_24hour = Window.partitionBy('ac_to').orderBy('trans_timestamp').rangeBetween(-86400, 0)
velocity_df = velocity_df.withColumn('senders_set_24hour', 
    collect_set('ac_from').over(window_recipient_24hour)) \
    .withColumn('unique_senders_24hour', 
    size(col('senders_set_24hour')))

velocity_df = velocity_df.withColumn('rule_velocity_multiple_senders_recipient', 
    when(col('unique_senders_24hour') > 5, 1).otherwise(0)) \
    .drop('senders_set_24hour')  # Remove intermediate column

print("   ✓ Cross-feature velocity rules applied")

# Calculate total velocity rules triggered
velocity_rule_cols = [c for c in velocity_df.columns if c.startswith('rule_velocity_')]
velocity_df = velocity_df.withColumn('velocity_rules_triggered', 
    sum([col(c) for c in velocity_rule_cols]))

print(f"✅ Velocity Rules Implemented: {len(velocity_rule_cols)} rules")
velocity_df.cache()
print(f"   • Transactions processed: {velocity_df.count():,}")

🚀 Implementing Velocity-Based Rules...
   • Rule 1.1: Transaction count velocity...
   ✓ Transaction count velocity rules applied
   • Rule 1.2: Amount velocity...
   ✓ Amount velocity rules applied
   • Rule 1.3: Cross-feature velocity...
   ✓ Cross-feature velocity rules applied
✅ Velocity Rules Implemented: 8 rules


ERROR:root:KeyboardInterrupt while sending command.============>(198 + 2) / 200]
Traceback (most recent call last):
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/root/miniconda3/envs/fraud/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/root/miniconda3/envs/fraud/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# RULE CATEGORY 2: TIME-BASED PATTERNS
print("\n🕐 Implementing Time-Based Pattern Rules...")

time_df = velocity_df

# Rule 2.1: High-Risk Time Windows
print("   • Rule 2.1: High-risk time windows...")

# Off-peak hours (1 AM - 5 AM)
time_df = time_df.withColumn('rule_time_offpeak_hours', 
    when((col('hour_of_day') >= 1) & (col('hour_of_day') < 5), 1).otherwise(0))

# Late night on weekends (11 PM - 6 AM)
time_df = time_df.withColumn('rule_time_weekend_night', 
    when((col('is_weekend') == 1) & 
         (((col('hour_of_day') >= 23) | (col('hour_of_day') < 6))), 1).otherwise(0))

# Night + Weekend combination (already exists as night_weekend_combo)
time_df = time_df.withColumn('rule_time_night_weekend_combo', 
    when(col('night_weekend_combo') == 1, 1).otherwise(0))

print("   ✓ High-risk time window rules applied")

# Rule 2.2: Unusual Time Patterns
print("   • Rule 2.2: Unusual time patterns...")

# First transaction during night hours (using user_first_txn_time to determine if this is first transaction)
time_df = time_df.withColumn('rule_time_first_txn_unusual', 
    when((col('trans_initiate_time') == col('user_first_txn_time')) & (col('is_night') == 1), 1).otherwise(0))

# Calculate typical transaction hour for each account (mode of hour_of_day)
# For simplicity, flag if transaction hour deviates significantly from usual pattern
# This requires historical pattern - for now, flag unusual hours as proxy
time_df = time_df.withColumn('rule_time_unusual_hour', 
    when(col('is_unusual_hour') == 1, 1).otherwise(0))

print("   ✓ Unusual time pattern rules applied")

# Calculate total time-based rules triggered
time_rule_cols = [c for c in time_df.columns if c.startswith('rule_time_')]
time_df = time_df.withColumn('time_rules_triggered', 
    sum([col(c) for c in time_rule_cols]))

print(f"✅ Time-Based Rules Implemented: {len(time_rule_cols)} rules")


🕐 Implementing Time-Based Pattern Rules...


NameError: name 'velocity_df' is not defined

In [ ]:
# RULE CATEGORY 3: ACCOUNT AGE / NEW ACCOUNT RULES
print("\n👶 Implementing Account Age Rules...")

age_df = time_df

# Calculate account age in days
age_df = age_df.withColumn('account_age_days', 
    datediff(col('cutoff_date'), col('mbar_registered_date_time')))

# Rule 3.1: Age-Based Risk Rules
print("   • Rule 3.1: Age-based risk rules...")

age_df = age_df.withColumn('rule_age_new_high_amount', 
    when((col('account_age_days') < 30) & (col('trx_amt') > 500), 1).otherwise(0)) \
    .withColumn('rule_age_very_new_any_txn', 
    when(col('account_age_days') < 7, 1).otherwise(0)) \
    .withColumn('rule_age_fresh_multiple_txns', 
    when((col('account_age_days') < 3) & (col('txn_count_24hour') > 1), 1).otherwise(0)) \
    .withColumn('rule_age_new_high_velocity', 
    when((col('account_age_days') < 30) & (col('txn_count_24hour') > 5), 1).otherwise(0))

print("   ✓ Age-based risk rules applied")

# Rule 3.2: Registration Time Patterns
print("   • Rule 3.2: Registration time patterns...")

# Extract hour from registration datetime
age_df = age_df.withColumn('registration_hour', 
    expr("hour(mbar_registered_date_time)"))

# Unusual registration time (midnight - 5 AM)
age_df = age_df.withColumn('rule_age_unusual_reg_time', 
    when((col('registration_hour') >= 0) & (col('registration_hour') < 5), 1).otherwise(0))

# First transaction within 1 hour of registration
age_df = age_df.withColumn('time_since_registration_hours', 
    (unix_timestamp(col('trans_initiate_time')) - unix_timestamp(col('mbar_registered_date_time'))) / 3600)

age_df = age_df.withColumn('rule_age_immediate_activity', 
    when((col('time_since_registration_hours') >= 0) & (col('time_since_registration_hours') < 1), 1).otherwise(0))

print("   ✓ Registration time pattern rules applied")

# Rule 3.3: Dormant-to-Active Pattern
print("   • Rule 3.3: Dormant account patterns...")

# Calculate days since last transaction
window_prev_txn = Window.partitionBy('ac_from').orderBy('trans_timestamp')
age_df = age_df.withColumn('prev_txn_timestamp', 
    lag('trans_timestamp', 1).over(window_prev_txn))

age_df = age_df.withColumn('days_since_last_txn', 
    (col('trans_timestamp') - col('prev_txn_timestamp')) / 86400)

# Dormant >30 days + High amount
age_df = age_df.withColumn('rule_age_dormant_30_high_amount', 
    when((col('days_since_last_txn') > 30) & (col('trx_amt') > 1000), 1).otherwise(0))

# Dormant >60 days + Multiple rapid transactions
age_df = age_df.withColumn('rule_age_dormant_60_rapid', 
    when((col('days_since_last_txn') > 60) & (col('txn_count_1hour') > 2), 1).otherwise(0))

print("   ✓ Dormant account pattern rules applied")

# Calculate total age-based rules triggered
age_rule_cols = [c for c in age_df.columns if c.startswith('rule_age_')]
age_df = age_df.withColumn('age_rules_triggered', 
    sum([col(c) for c in age_rule_cols]))

print(f"✅ Account Age Rules Implemented: {len(age_rule_cols)} rules")


👶 Implementing Account Age Rules...


NameError: name 'time_df' is not defined

In [ ]:
# RULE CATEGORY 4: AMOUNT-BASED RULES
print("\n💰 Implementing Amount-Based Rules...")

amount_df = age_df

# Rule 4.1: Threshold Rules
print("   • Rule 4.1: Threshold rules...")

# Calculate percentile and standard deviation (using window over account history)
window_account_stats = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-100, -1)

amount_df = amount_df.withColumn('historical_avg_amount', 
    coalesce(avg('trx_amt').over(window_account_stats), lit(0))) \
    .withColumn('historical_stddev_amount', 
    coalesce(stddev('trx_amt').over(window_account_stats), lit(0)))

# Amount > 95th percentile (approximate as mean + 2*stddev)
amount_df = amount_df.withColumn('rule_amount_95_percentile', 
    when((col('trx_amt') > col('historical_avg_amount') + 2 * col('historical_stddev_amount')) & 
         (col('historical_stddev_amount') > 0), 1).otherwise(0))

# Amount > 2-3 standard deviations
amount_df = amount_df.withColumn('rule_amount_3_stddev', 
    when((col('trx_amt') > col('historical_avg_amount') + 3 * col('historical_stddev_amount')) & 
         (col('historical_stddev_amount') > 0), 1).otherwise(0))

# High-value threshold
amount_df = amount_df.withColumn('rule_amount_high_value', 
    when(col('trx_amt') > 5000, 1).otherwise(0))

# Round number amounts (exactly divisible by 1000)
amount_df = amount_df.withColumn('rule_amount_round_number', 
    when((col('trx_amt') % 1000 == 0) & (col('trx_amt') >= 1000), 1).otherwise(0))

print("   ✓ Threshold rules applied")

# Rule 4.2: Pattern Rules
print("   • Rule 4.2: Pattern rules...")

# Multiple transactions with uniform amounts (check last 5 transactions)
window_last_5_txns = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-5, -1)
amount_df = amount_df.withColumn('stddev_last_5_amounts', 
    coalesce(stddev('trx_amt').over(window_last_5_txns), lit(999999)))

amount_df = amount_df.withColumn('rule_amount_uniform_pattern', 
    when((col('stddev_last_5_amounts') < 10) & (col('stddev_last_5_amounts') > 0), 1).otherwise(0))

# Just-below-threshold amounts (structuring)
amount_df = amount_df.withColumn('rule_amount_structuring', 
    when(((col('trx_amt') >= 4900) & (col('trx_amt') < 5000)) | 
         ((col('trx_amt') >= 9900) & (col('trx_amt') < 10000)), 1).otherwise(0))

# Amount spike (>300% of typical)
amount_df = amount_df.withColumn('rule_amount_spike', 
    when((col('trx_amt') > col('historical_avg_amount') * 3) & 
         (col('historical_avg_amount') > 0), 1).otherwise(0))

print("   ✓ Pattern rules applied")

# Calculate total amount-based rules triggered
amount_rule_cols = [c for c in amount_df.columns if c.startswith('rule_amount_')]
amount_df = amount_df.withColumn('amount_rules_triggered', 
    sum([col(c) for c in amount_rule_cols]))

print(f"✅ Amount-Based Rules Implemented: {len(amount_rule_cols)} rules")


💰 Implementing Amount-Based Rules...


NameError: name 'age_df' is not defined

In [ ]:
# RULE CATEGORY 5: CHANNEL-BASED RULES
print("\n📱 Implementing Channel-Based Rules...")

channel_df = amount_df

# Rule 5.1: High-Risk Channel Patterns
print("   • Rule 5.1: High-risk channel patterns...")

# Payment Gateway + New account + High amount
channel_df = channel_df.withColumn('rule_channel_pgw_new_high', 
    when((col('trx_channel').isin(['Payment Gateway', 'PGW'])) & 
         (col('account_age_days') < 30) & 
         (col('trx_amt') > 1000), 1).otherwise(0))

# Mobile App + New account + High amount
channel_df = channel_df.withColumn('rule_channel_mobile_new_high', 
    when((col('trx_channel').isin(['Mobile App', 'NEW_JC_APP'])) & 
         (col('account_age_days') < 30) & 
         (col('trx_amt') > 1000), 1).otherwise(0))

# Channel switch detection (compare current channel with most used channel)
# Using user_most_used_channel_7d from features if available
if 'user_most_used_channel_7d' in channel_df.columns:
    channel_df = channel_df.withColumn('rule_channel_switch', 
        when((col('trx_channel') != col('user_most_used_channel_7d')) & 
             (col('user_most_used_channel_7d').isNotNull()), 1).otherwise(0))
else:
    channel_df = channel_df.withColumn('rule_channel_switch', lit(0))

print("   ✓ High-risk channel pattern rules applied")

# Rule 5.2: Channel Velocity
print("   • Rule 5.2: Channel velocity...")

# Already calculated: channels_used_30min and rule_velocity_multiple_channels
# Add: Same channel, multiple accounts (requires cross-account analysis)
# This is complex - for now, flag high-risk channels with high transaction counts

channel_df = channel_df.withColumn('rule_channel_high_velocity', 
    when(col('txn_count_15min') > 5, 1).otherwise(0))

print("   ✓ Channel velocity rules applied")

# Calculate total channel-based rules triggered
channel_rule_cols = [c for c in channel_df.columns if c.startswith('rule_channel_')]
channel_df = channel_df.withColumn('channel_rules_triggered', 
    sum([col(c) for c in channel_rule_cols]))

print(f"✅ Channel-Based Rules Implemented: {len(channel_rule_cols)} rules")


📱 Implementing Channel-Based Rules...


NameError: name 'amount_df' is not defined

In [ ]:
# RULE CATEGORY 6: ACCOUNT TYPE RULES
print("\n🏦 Implementing Account Type Rules...")

# Note: mbar_account_type_name is already filtered to 'Customer Account'
# This section would be more relevant with multiple account types

actype_df = channel_df

# Placeholder rules - these would be more useful with multiple account types
print("   • Account type rules (limited - all Customer Accounts)...")

# For future: Cross-type transaction patterns
# For now, mark as placeholder
actype_df = actype_df.withColumn('rule_actype_placeholder', lit(0))

actype_df = actype_df.withColumn('actype_rules_triggered', lit(0))

print("✅ Account Type Rules: Placeholder (all transactions are Customer Accounts)")


🏦 Implementing Account Type Rules...


NameError: name 'channel_df' is not defined

In [ ]:
# RULE CATEGORY 7: BEHAVIORAL ANOMALY RULES
print("\n🎯 Implementing Behavioral Anomaly Rules...")

behavior_df = actype_df

# Rule 7.1: Deviation from Normal Patterns
print("   • Rule 7.1: Deviation from normal patterns...")

# Import row_number function
from pyspark.sql.functions import row_number

# First transaction to new recipient with high amount
window_recipient_history = Window.partitionBy('ac_from', 'ac_to').orderBy('trans_timestamp')
behavior_df = behavior_df.withColumn('is_first_to_recipient', 
    when(row_number().over(window_recipient_history) == 1, 1).otherwise(0))

behavior_df = behavior_df.withColumn('rule_behavior_new_recipient_high', 
    when((col('is_first_to_recipient') == 1) & (col('trx_amt') > 1000), 1).otherwise(0))

# Weekend activity for typically weekday users
# Check if user typically transacts on weekdays
window_user_history = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-50, -1)
behavior_df = behavior_df.withColumn('historical_weekend_ratio', 
    coalesce(avg('is_weekend').over(window_user_history), lit(0.5)))

behavior_df = behavior_df.withColumn('rule_behavior_unusual_weekend', 
    when((col('is_weekend') == 1) & 
         (col('historical_weekend_ratio') < 0.2) & 
         (col('trx_amt') > 500), 1).otherwise(0))

# Night activity for typically daytime users
behavior_df = behavior_df.withColumn('historical_night_ratio', 
    coalesce(avg('is_night').over(window_user_history), lit(0.3)))

behavior_df = behavior_df.withColumn('rule_behavior_unusual_night', 
    when((col('is_night') == 1) & 
         (col('historical_night_ratio') < 0.1) & 
         (col('trx_amt') > 500), 1).otherwise(0))

print("   ✓ Behavioral deviation rules applied")

# Rule 7.2: Sudden Pattern Changes
print("   • Rule 7.2: Sudden pattern changes...")

# Sudden amount increase
behavior_df = behavior_df.withColumn('rule_behavior_amount_jump', 
    when((col('trx_amt') > col('historical_avg_amount') * 5) & 
         (col('historical_avg_amount') > 0) & 
         (col('historical_avg_amount') < 1000), 1).otherwise(0))

# Sudden velocity increase
window_prev_period = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-100, -50)
behavior_df = behavior_df.withColumn('prev_period_avg_daily_txns', 
    coalesce(count('*').over(window_prev_period) / 50.0, lit(0)))

behavior_df = behavior_df.withColumn('rule_behavior_velocity_jump', 
    when((col('txn_count_24hour') > col('prev_period_avg_daily_txns') * 3) & 
         (col('prev_period_avg_daily_txns') > 0), 1).otherwise(0))

print("   ✓ Pattern change rules applied")

# Calculate total behavioral rules triggered
behavior_rule_cols = [c for c in behavior_df.columns if c.startswith('rule_behavior_')]
behavior_df = behavior_df.withColumn('behavior_rules_triggered', 
    sum([col(c) for c in behavior_rule_cols]))

print(f"✅ Behavioral Anomaly Rules Implemented: {len(behavior_rule_cols)} rules")


🎯 Implementing Behavioral Anomaly Rules...


NameError: name 'actype_df' is not defined

In [ ]:
# RULE CATEGORY 8: RECIPIENT/BENEFICIARY PATTERNS
print("\n👥 Implementing Recipient/Beneficiary Rules...")

recipient_df = behavior_df

# Rule 8.1: High-Risk Recipients
print("   • Rule 8.1: High-risk recipient patterns...")

# Calculate recipient account age (if we have registration data for recipients)
# For now, flag new recipients with high amounts

# New recipient + High amount + Sender is new account
recipient_df = recipient_df.withColumn('rule_recipient_new_high_amount', 
    when((col('is_first_to_recipient') == 1) & 
         (col('trx_amt') > 1000) & 
         (col('account_age_days') < 30), 1).otherwise(0))

# Same recipient from multiple senders (money mule indicator)
# Already calculated: unique_senders_24hour and rule_velocity_multiple_senders_recipient
recipient_df = recipient_df.withColumn('rule_recipient_money_mule', 
    when(col('unique_senders_24hour') > 10, 1).otherwise(0))

# High amount to recipient in short time
window_recipient_1hour = Window.partitionBy('ac_to').orderBy('trans_timestamp').rangeBetween(-3600, 0)
recipient_df = recipient_df.withColumn('recipient_amount_1hour', 
    spark_sum('trx_amt').over(window_recipient_1hour))

recipient_df = recipient_df.withColumn('rule_recipient_high_amount_1hour', 
    when(col('recipient_amount_1hour') > 10000, 1).otherwise(0))

print("   ✓ Recipient pattern rules applied")

# Calculate total recipient rules triggered
recipient_rule_cols = [c for c in recipient_df.columns if c.startswith('rule_recipient_')]
recipient_df = recipient_df.withColumn('recipient_rules_triggered', 
    sum([col(c) for c in recipient_rule_cols]))

print(f"✅ Recipient/Beneficiary Rules Implemented: {len(recipient_rule_cols)} rules")


👥 Implementing Recipient/Beneficiary Rules...


NameError: name 'behavior_df' is not defined

In [ ]:
# RULE CATEGORY 9: FRAUD RING DETECTION
print("\n🕸️  Implementing Fraud Ring Detection Rules...")

# First, we need to identify high-risk accounts from fraud data
# This would typically come from analysis of historical fraud

# Step 1: Identify accounts with multiple fraud transactions
print("   • Step 1: Identifying high-risk accounts...")

fraud_account_counts = recipient_df.filter(col('fraud_flag') == 1).groupBy('ac_to').agg(
    count('*').alias('fraud_count')
).filter(col('fraud_count') >= 10)

high_risk_accounts = fraud_account_counts.select('ac_from').distinct()
high_risk_account_list = [row.ac_from for row in high_risk_accounts.collect()]

print(f"   ✓ Identified {len(high_risk_account_list)} high-risk accounts (10+ frauds)")

# Step 2: Create fraud ring rules
ring_df = recipient_df

# Rule 9.1: Transaction from/to known fraud ring accounts
print("   • Rule 9.1: Known fraud ring member detection...")

if len(high_risk_account_list) > 0:
    # Broadcast the list for efficient lookup
    from pyspark.sql.functions import array, lit as sql_lit
    
    ring_df = ring_df.withColumn('rule_ring_known_fraudster_sender', 
        when(col('ac_from').isin(high_risk_account_list), 1).otherwise(0))
    
    ring_df = ring_df.withColumn('rule_ring_known_fraudster_recipient', 
        when(col('ac_to').isin(high_risk_account_list), 1).otherwise(0))
else:
    ring_df = ring_df.withColumn('rule_ring_known_fraudster_sender', lit(0))
    ring_df = ring_df.withColumn('rule_ring_known_fraudster_recipient', lit(0))

print("   ✓ Known fraud ring rules applied")

# Rule 9.2: Network patterns (shared recipients/senders)
print("   • Rule 9.2: Network pattern detection...")

# Accounts that share many common recipients with known fraudsters
# This requires complex graph analysis - for now, use velocity indicators as proxy

ring_df = ring_df.withColumn('rule_ring_high_recipient_network', 
    when((col('unique_senders_24hour') > 15) | (col('txn_count_24hour') > 15), 1).otherwise(0))

# Multiple accounts with similar transaction patterns (same amounts, same times)
# This is complex - flag accounts with highly structured transaction patterns
ring_df = ring_df.withColumn('rule_ring_structured_pattern', 
    when((col('rule_amount_uniform_pattern') == 1) & 
         (col('txn_count_24hour') > 10), 1).otherwise(0))

print("   ✓ Network pattern rules applied")

# Rule 9.3: Coordinated activity indicators
print("   • Rule 9.3: Coordinated activity detection...")

# Multiple transactions at exact same time (within 1 minute)
# Multiple accounts using same channel at same time with similar amounts
# This requires cross-account analysis - use high velocity as proxy

ring_df = ring_df.withColumn('rule_ring_coordinated_activity', 
    when((col('txn_count_15min') > 8) & (col('rule_amount_uniform_pattern') == 1), 1).otherwise(0))

print("   ✓ Coordinated activity rules applied")

# Calculate total fraud ring rules triggered
ring_rule_cols = [c for c in ring_df.columns if c.startswith('rule_ring_')]
ring_df = ring_df.withColumn('ring_rules_triggered', 
    sum([col(c) for c in ring_rule_cols]))

print(f"✅ Fraud Ring Detection Rules Implemented: {len(ring_rule_cols)} rules")


🕸️  Implementing Fraud Ring Detection Rules...
   • Step 1: Identifying high-risk accounts...


NameError: name 'recipient_df' is not defined

In [ ]:
# CALCULATE OVERALL FRAUD RISK SCORE
print("\n🎯 Calculating Overall Fraud Risk Scores...")

fraud_rules_df = ring_df

# Calculate total rules triggered across all categories
rule_category_cols = [
    'velocity_rules_triggered',
    'time_rules_triggered', 
    'age_rules_triggered',
    'amount_rules_triggered',
    'channel_rules_triggered',
    'actype_rules_triggered',
    'behavior_rules_triggered',
    'recipient_rules_triggered',
    'ring_rules_triggered'
]

fraud_rules_df = fraud_rules_df.withColumn('total_rules_triggered', 
    sum([col(c) for c in rule_category_cols]))

# Calculate weighted risk score (some rules are more important)
fraud_rules_df = fraud_rules_df.withColumn('fraud_risk_score',
    (col('velocity_rules_triggered') * 3.0) +      # High weight
    (col('time_rules_triggered') * 1.5) +
    (col('age_rules_triggered') * 2.0) +           # High weight for new accounts
    (col('amount_rules_triggered') * 2.5) +        # High weight
    (col('channel_rules_triggered') * 1.5) +
    (col('actype_rules_triggered') * 1.0) +
    (col('behavior_rules_triggered') * 2.0) +
    (col('recipient_rules_triggered') * 2.5) +
    (col('ring_rules_triggered') * 5.0)            # Highest weight - known fraud rings
)

# Assign risk categories
fraud_rules_df = fraud_rules_df.withColumn('risk_category',
    when(col('fraud_risk_score') >= 20, 'CRITICAL') \
    .when(col('fraud_risk_score') >= 15, 'HIGH') \
    .when(col('fraud_risk_score') >= 10, 'MEDIUM') \
    .when(col('fraud_risk_score') >= 5, 'LOW') \
    .otherwise('MINIMAL')
)

# Cache the final result
fraud_rules_df = fraud_rules_df.coalesce(100).cache()

print("✅ Fraud risk scores calculated!")
print("\n" + "="*80)
print("FRAUD DETECTION RULES ENGINE - SUMMARY")
print("="*80)

total_txns = fraud_rules_df.count()
print(f"Total transactions processed: {total_txns:,}")

# Count all rule columns
all_rule_cols = [c for c in fraud_rules_df.columns if c.startswith('rule_')]
print(f"Total rules implemented: {len(all_rule_cols)}")
print("\nRules by category:")
print(f"  • Velocity rules: {len([c for c in all_rule_cols if 'velocity' in c])}")
print(f"  • Time-based rules: {len([c for c in all_rule_cols if 'time' in c])}")
print(f"  • Account age rules: {len([c for c in all_rule_cols if 'age' in c])}")
print(f"  • Amount rules: {len([c for c in all_rule_cols if 'amount' in c])}")
print(f"  • Channel rules: {len([c for c in all_rule_cols if 'channel' in c])}")
print(f"  • Behavioral rules: {len([c for c in all_rule_cols if 'behavior' in c])}")
print(f"  • Recipient rules: {len([c for c in all_rule_cols if 'recipient' in c])}")
print(f"  • Fraud ring rules: {len([c for c in all_rule_cols if 'ring' in c])}")

print("\n" + "="*80)


🎯 Calculating Overall Fraud Risk Scores...


NameError: name 'ring_df' is not defined

In [ ]:
# ANALYZE RULE PERFORMANCE
print("\n📊 Analyzing Rule Performance...")

# Risk category distribution
print("\n🎯 Risk Category Distribution:")
fraud_rules_df.groupBy('risk_category', 'fraud_flag').count().orderBy('risk_category', 'fraud_flag').show()

# Top triggered rules
print("\n🔥 Top 10 Most Triggered Rules:")
rule_trigger_counts = []
for rule_col in all_rule_cols:
    trigger_count = fraud_rules_df.filter(col(rule_col) == 1).count()
    rule_trigger_counts.append((rule_col, trigger_count))

rule_trigger_counts.sort(key=lambda x: x[1], reverse=True)
for rule, count in rule_trigger_counts[:10]:
    print(f"  {rule}: {count:,} ({count/total_txns*100:.2f}%)")

# Rules triggered distribution
print("\n📈 Total Rules Triggered Distribution:")
fraud_rules_df.groupBy('total_rules_triggered').count().orderBy('total_rules_triggered').show(20)b

SyntaxError: invalid syntax (2209865153.py, line 21)

In [ ]:

# Fraud detection by risk category
from pyspark.sql.functions import col, count

print("\n🎯 Fraud Detection by Risk Category:")
fraud_rules_df.groupBy('risk_category').agg(
    count('*').alias('total_txns'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_round((spark_sum('fraud_flag') / count('*')) * 100, 2).alias('fraud_rate_%')
).orderBy('risk_category').show()

# Average risk score by fraud flag
print("\n📊 Average Risk Score by Fraud Flag:")
fraud_rules_df.groupBy('fraud_flag').agg(
    count('*').alias('count'),
    spark_round(avg('fraud_risk_score'), 2).alias('avg_risk_score'),
    spark_round(stddev('fraud_risk_score'), 2).alias('stddev_risk_score'),
    spark_min('fraud_risk_score').alias('min_risk_score'),
    spark_max('fraud_risk_score').alias('max_risk_score')
).show()

# Rule category effectiveness
print("\n🎯 Rule Category Effectiveness (Average triggers for fraud vs non-fraud):")
for cat_col in rule_category_cols:
    print(f"\n{cat_col}:")
    fraud_rules_df.groupBy('fraud_flag').agg(
        spark_round(avg(cat_col), 2).alias(f'avg_{cat_col}')
    ).orderBy('fraud_flag').show()


🎯 Fraud Detection by Risk Category:


NameError: name 'fraud_rules_df' is not defined

In [ ]:
# HIGH-RISK TRANSACTION ANALYSIS
print("\n🚨 Analyzing High-Risk Transactions...")

# Filter high-risk transactions
high_risk_txns = fraud_rules_df.filter(col('risk_category').isin(['HIGH', 'CRITICAL']))

print(f"\n📊 High-Risk Transaction Summary:")
print(f"  • Total high-risk transactions: {high_risk_txns.count():,}")
print(f"  • Percentage of all transactions: {high_risk_txns.count()/total_txns*100:.2f}%")

high_risk_fraud = high_risk_txns.filter(col('fraud_flag') == 1).count()
print(f"  • Actual frauds in high-risk: {high_risk_fraud:,}")
print(f"  • Fraud detection rate: {high_risk_fraud/high_risk_txns.count()*100:.2f}%")

# Top channels for high-risk transactions
print("\n📱 Top Channels in High-Risk Transactions:")
high_risk_txns.groupBy('trx_channel').agg(
    count('*').alias('count'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_round(avg('fraud_risk_score'), 2).alias('avg_risk_score')
).orderBy(col('count').desc()).show(15, truncate=False)

# Top transaction types for high-risk
print("\n💳 Top Transaction Types in High-Risk Transactions:")
high_risk_txns.groupBy('trx_type').agg(
    count('*').alias('count'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_round(avg('fraud_risk_score'), 2).alias('avg_risk_score')
).orderBy(col('count').desc()).show(15, truncate=False)

# Time patterns in high-risk transactions
print("\n🕐 Time Patterns in High-Risk Transactions:")
high_risk_txns.groupBy('hour_of_day').agg(
    count('*').alias('count'),
    spark_sum('fraud_flag').alias('fraud_count')
).orderBy('hour_of_day').show(24)


🚨 Analyzing High-Risk Transactions...


NameError: name 'fraud_rules_df' is not defined

In [ ]:
# SAMPLE HIGH-RISK TRANSACTIONS
print("\n📋 Sample High-Risk Fraud Transactions:")

high_risk_fraud_sample = fraud_rules_df.filter(
    (col('risk_category').isin(['HIGH', 'CRITICAL'])) & 
    (col('fraud_flag') == 1)
).select(
    'trans_id', 'ac_from', 'ac_to', 'cutoff_date', 'trx_channel', 'trx_type', 
    'trx_amt', 'fraud_risk_score', 'risk_category', 'total_rules_triggered',
    'velocity_rules_triggered', 'amount_rules_triggered', 'ring_rules_triggered'
).orderBy(col('fraud_risk_score').desc())

print(f"\nTop 20 Highest Risk Fraud Transactions:")
high_risk_fraud_sample.show(20, truncate=False)

print("\n📋 Sample High-Risk False Positives (flagged but not fraud):")
high_risk_false_positives = fraud_rules_df.filter(
    (col('risk_category').isin(['HIGH', 'CRITICAL'])) & 
    (col('fraud_flag') == 0)
).select(
    'trans_id', 'ac_from', 'ac_to', 'cutoff_date', 'trx_channel', 'trx_type', 
    'trx_amt', 'fraud_risk_score', 'risk_category', 'total_rules_triggered',
    'velocity_rules_triggered', 'amount_rules_triggered'
).orderBy(col('fraud_risk_score').desc())

print(f"\nTop 20 False Positives (for rule tuning):")
high_risk_false_positives.show(20, truncate=False)


📋 Sample High-Risk Fraud Transactions:


NameError: name 'fraud_rules_df' is not defined

In [35]:
# SAVE FRAUD RULES RESULTS
save_fraud_rules = False  # Set to True to save

if save_fraud_rules:
    print("\n💾 Saving fraud rules results...")
    
    # Save full dataset with all rules
    rules_output_path = f"/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_results_{sample_start_date}_to_{sample_end_date}"
    fraud_rules_df.write.mode('overwrite').parquet(rules_output_path)
    print(f"✅ Full results saved to: {rules_output_path}")
    
    # Save high-risk transactions only
    high_risk_output_path = f"/root/research-dir/dev/jazzcash-fraud-detection/data/high_risk_transactions_{sample_start_date}_to_{sample_end_date}"
    high_risk_txns.write.mode('overwrite').parquet(high_risk_output_path)
    print(f"✅ High-risk transactions saved to: {high_risk_output_path}")
    
    # Save rule performance summary
    summary_output = f"/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_summary_{sample_start_date}_to_{sample_end_date}.txt"
    with open(summary_output, 'w') as f:
        f.write("FRAUD DETECTION RULES ENGINE - PERFORMANCE SUMMARY\n")
        f.write("="*80 + "\n\n")
        f.write(f"Date Range: {sample_start_date} to {sample_end_date}\n")
        f.write(f"Total Transactions: {total_txns:,}\n")
        f.write(f"Total Rules Implemented: {len(all_rule_cols)}\n\n")
        f.write(f"High-Risk Transactions: {high_risk_txns.count():,}\n")
        f.write(f"High-Risk Fraud Detection Rate: {high_risk_fraud/high_risk_txns.count()*100:.2f}%\n\n")
        f.write("Rule Categories:\n")
        for cat in rule_category_cols:
            f.write(f"  - {cat}\n")
    print(f"✅ Summary saved to: {summary_output}")
    
    # Export high-risk accounts list
    high_risk_accounts_df = fraud_rules_df.filter(
        col('risk_category').isin(['HIGH', 'CRITICAL'])
    ).select('ac_from').distinct()
    
    high_risk_accounts_path = f"/root/research-dir/dev/jazzcash-fraud-detection/data/high_risk_accounts_{sample_start_date}_to_{sample_end_date}.csv"
    high_risk_accounts_df.toPandas().to_csv(high_risk_accounts_path, index=False)
    print(f"✅ High-risk accounts list saved to: {high_risk_accounts_path}")
    
else:
    print("\nℹ️  Set save_fraud_rules=True to save results")
    print("   This will save:")
    print("   • Full dataset with all rule flags and scores (Parquet)")
    print("   • High-risk transactions subset (Parquet)")
    print("   • Performance summary (Text)")
    print("   • High-risk accounts list (CSV)")

print("\n" + "="*80)
print("✅ FRAUD DETECTION RULES ENGINE COMPLETE!")
print("="*80)
print(f"\n📊 Summary Statistics:")
print(f"  • Total transactions analyzed: {total_txns:,}")
print(f"  • Total rules implemented: {len(all_rule_cols)}")
print(f"  • High-risk transactions flagged: {high_risk_txns.count():,}")
print(f"  • High-risk fraud detection rate: {high_risk_fraud/high_risk_txns.count()*100:.2f}%")
print(f"\n🎯 Next Steps:")
print(f"  1. Review false positives to tune rule thresholds")
print(f"  2. Analyze missed frauds (low risk score but fraud_flag=1)")
print(f"  3. Implement real-time scoring API")
print(f"  4. Set up automated alerts for CRITICAL risk transactions")
print(f"  5. Create rule performance dashboard")


ℹ️  Set save_fraud_rules=True to save results
   This will save:
   • Full dataset with all rule flags and scores (Parquet)
   • High-risk transactions subset (Parquet)
   • Performance summary (Text)
   • High-risk accounts list (CSV)

✅ FRAUD DETECTION RULES ENGINE COMPLETE!

📊 Summary Statistics:
  • Total transactions analyzed: 3,790,938
  • Total rules implemented: 46
  • High-risk transactions flagged: 1,322,458
  • High-risk fraud detection rate: 0.63%

🎯 Next Steps:
  1. Review false positives to tune rule thresholds
  2. Analyze missed frauds (low risk score but fraud_flag=1)
  3. Implement real-time scoring API
  4. Set up automated alerts for CRITICAL risk transactions
  5. Create rule performance dashboard


## Rule Performance Evaluation: Precision & False Positive Rate

Calculate precision and false positive rate for each individual fraud detection rule to identify the most effective rules.

In [36]:
# Calculate Precision and False Positive Rate for each rule
print("📊 Calculating Precision and False Positive Rate for each rule...\n")

import pandas as pd

# Get total fraud and non-fraud counts
total_fraud = fraud_rules_df.filter(col('fraud_flag') == 1).count()
total_non_fraud = fraud_rules_df.filter(col('fraud_flag') == 0).count()

print(f"Dataset Summary:")
print(f"  • Total frauds: {total_fraud:,}")
print(f"  • Total non-frauds: {total_non_fraud:,}")
print(f"  • Total transactions: {total_txns:,}")
print(f"  • Baseline fraud rate: {(total_fraud/total_txns)*100:.2f}%\n")

# Calculate metrics for each rule
rule_performance = []

for rule_col in all_rule_cols:
    # Get counts
    rule_triggered = fraud_rules_df.filter(col(rule_col) == 1)
    rule_triggered_count = rule_triggered.count()
    
    if rule_triggered_count == 0:
        continue  # Skip rules that never triggered
    
    # True Positives: Rule triggered AND fraud
    true_positives = rule_triggered.filter(col('fraud_flag') == 1).count()
    
    # False Positives: Rule triggered BUT NOT fraud
    false_positives = rule_triggered.filter(col('fraud_flag') == 0).count()
    
    # False Negatives: Rule NOT triggered BUT fraud
    false_negatives = fraud_rules_df.filter((col(rule_col) == 0) & (col('fraud_flag') == 1)).count()
    
    # True Negatives: Rule NOT triggered AND NOT fraud
    true_negatives = fraud_rules_df.filter((col(rule_col) == 0) & (col('fraud_flag') == 0)).count()
    
    # Calculate metrics
    precision = (true_positives / rule_triggered_count * 100) if rule_triggered_count > 0 else 0
    recall = (true_positives / total_fraud * 100) if total_fraud > 0 else 0
    false_positive_rate = (false_positives / total_non_fraud * 100) if total_non_fraud > 0 else 0
    
    # F1 Score
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
    
    # Lift: How much better than random
    lift = (precision / (total_fraud/total_txns*100)) if total_fraud > 0 else 0
    
    rule_performance.append({
        'rule_name': rule_col,
        'times_triggered': rule_triggered_count,
        'trigger_rate_%': round(rule_triggered_count / total_txns * 100, 2),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'true_negatives': true_negatives,
        'precision_%': round(precision, 2),
        'recall_%': round(recall, 2),
        'false_positive_rate_%': round(false_positive_rate, 2),
        'f1_score': round(f1_score, 2),
        'lift': round(lift, 2)
    })

# Convert to pandas DataFrame for analysis
rule_perf_df = pd.DataFrame(rule_performance)

print(f"✅ Calculated metrics for {len(rule_performance)} rules")
print(f"   (Skipped {len(all_rule_cols) - len(rule_performance)} rules that never triggered)\n")

📊 Calculating Precision and False Positive Rate for each rule...

Dataset Summary:
  • Total frauds: 13,753
  • Total non-frauds: 3,777,185
  • Total transactions: 3,790,938
  • Baseline fraud rate: 0.36%

✅ Calculated metrics for 41 rules
   (Skipped 5 rules that never triggered)



In [37]:
# Sort rules by different metrics
print("="*100)
print("TOP 20 RULES BY PRECISION (Accuracy when rule triggers)")
print("="*100)
print(rule_perf_df.sort_values('precision_%', ascending=False).head(20).to_string(index=False))

print("\n" + "="*100)
print("TOP 20 RULES BY RECALL (Fraud Detection Coverage)")
print("="*100)
print(rule_perf_df.sort_values('recall_%', ascending=False).head(20).to_string(index=False))

print("\n" + "="*100)
print("TOP 20 RULES BY F1 SCORE (Balance of Precision and Recall)")
print("="*100)
print(rule_perf_df.sort_values('f1_score', ascending=False).head(20).to_string(index=False))

print("\n" + "="*100)
print("TOP 20 RULES BY LIFT (Improvement over Random)")
print("="*100)
print(rule_perf_df.sort_values('lift', ascending=False).head(20).to_string(index=False))

TOP 20 RULES BY PRECISION (Accuracy when rule triggers)
                           rule_name  times_triggered  trigger_rate_%  true_positives  false_positives  false_negatives  true_negatives  precision_%  recall_%  false_positive_rate_%  f1_score   lift
    rule_ring_known_fraudster_sender              553            0.01             543               10            13210         3777175        98.19      3.95                   0.00      7.59 270.66
           rule_velocity_count_15min              690            0.02             601               89            13152         3777096        87.10      4.37                   0.00      8.32 240.09
           rule_velocity_count_1hour              448            0.01             377               71            13376         3777114        84.15      2.74                   0.00      5.31 231.96
          rule_channel_high_velocity              356            0.01             289               67            13464         3777118        81.18

In [38]:
# Rules with LOWEST False Positive Rate (most accurate)
print("\n" + "="*100)
print("TOP 20 RULES BY LOWEST FALSE POSITIVE RATE (Most Accurate)")
print("="*100)
print(rule_perf_df.sort_values('false_positive_rate_%', ascending=True).head(20).to_string(index=False))

print("\n" + "="*100)
print("BOTTOM 20 RULES BY FALSE POSITIVE RATE (Highest False Alarms)")
print("="*100)
print(rule_perf_df.sort_values('false_positive_rate_%', ascending=False).head(20).to_string(index=False))


TOP 20 RULES BY LOWEST FALSE POSITIVE RATE (Most Accurate)
                          rule_name  times_triggered  trigger_rate_%  true_positives  false_positives  false_negatives  true_negatives  precision_%  recall_%  false_positive_rate_%  f1_score   lift
          rule_velocity_count_15min              690            0.02             601               89            13152         3777096        87.10      4.37                   0.00      8.32 240.09
          rule_velocity_count_1hour              448            0.01             377               71            13376         3777114        84.15      2.74                   0.00      5.31 231.96
         rule_velocity_count_24hour              309            0.01             221               88            13532         3777097        71.52      1.61                   0.00      3.14 197.14
         rule_age_new_high_velocity               12            0.00               3                9            13750         3777176        25.00 

In [39]:
# Categorize rules by performance
print("\n" + "="*100)
print("RULE CATEGORIZATION BY PERFORMANCE")
print("="*100)

# Excellent rules: High precision, Low FPR, Good recall
excellent_rules = rule_perf_df[
    (rule_perf_df['precision_%'] >= 80) & 
    (rule_perf_df['false_positive_rate_%'] < 5) & 
    (rule_perf_df['recall_%'] >= 5)
]

# Good rules: Good precision, Moderate FPR
good_rules = rule_perf_df[
    (rule_perf_df['precision_%'] >= 60) & 
    (rule_perf_df['false_positive_rate_%'] < 10) & 
    (rule_perf_df['recall_%'] >= 3) &
    (~rule_perf_df['rule_name'].isin(excellent_rules['rule_name']))
]

# Moderate rules: Decent balance
moderate_rules = rule_perf_df[
    (rule_perf_df['precision_%'] >= 40) & 
    (rule_perf_df['false_positive_rate_%'] < 20) &
    (~rule_perf_df['rule_name'].isin(excellent_rules['rule_name'])) &
    (~rule_perf_df['rule_name'].isin(good_rules['rule_name']))
]

# Poor rules: Low precision or High FPR
poor_rules = rule_perf_df[
    (rule_perf_df['precision_%'] < 40) | 
    (rule_perf_df['false_positive_rate_%'] >= 20)
]

print(f"\n🌟 EXCELLENT RULES ({len(excellent_rules)} rules):")
print(f"   (Precision ≥80%, FPR <5%, Recall ≥5%)")
if len(excellent_rules) > 0:
    print(excellent_rules[['rule_name', 'precision_%', 'recall_%', 'false_positive_rate_%', 'f1_score']].to_string(index=False))
else:
    print("   None found")

print(f"\n✅ GOOD RULES ({len(good_rules)} rules):")
print(f"   (Precision ≥60%, FPR <10%, Recall ≥3%)")
if len(good_rules) > 0:
    print(good_rules[['rule_name', 'precision_%', 'recall_%', 'false_positive_rate_%', 'f1_score']].to_string(index=False))
else:
    print("   None found")

print(f"\n⚠️  MODERATE RULES ({len(moderate_rules)} rules):")
print(f"   (Precision ≥40%, FPR <20%)")
if len(moderate_rules) > 0:
    print(moderate_rules.head(10)[['rule_name', 'precision_%', 'recall_%', 'false_positive_rate_%', 'f1_score']].to_string(index=False))
else:
    print("   None found")

print(f"\n❌ POOR RULES ({len(poor_rules)} rules):")
print(f"   (Precision <40% OR FPR ≥20%) - Consider removing or tuning")
if len(poor_rules) > 0:
    print(poor_rules.head(10)[['rule_name', 'precision_%', 'recall_%', 'false_positive_rate_%', 'f1_score']].to_string(index=False))
else:
    print("   None found")


RULE CATEGORIZATION BY PERFORMANCE

🌟 EXCELLENT RULES (0 rules):
   (Precision ≥80%, FPR <5%, Recall ≥5%)
   None found

✅ GOOD RULES (2 rules):
   (Precision ≥60%, FPR <10%, Recall ≥3%)
                       rule_name  precision_%  recall_%  false_positive_rate_%  f1_score
       rule_velocity_count_15min        87.10      4.37                    0.0      8.32
rule_ring_known_fraudster_sender        98.19      3.95                    0.0      7.59

⚠️  MODERATE RULES (3 rules):
   (Precision ≥40%, FPR <20%)
                 rule_name  precision_%  recall_%  false_positive_rate_%  f1_score
 rule_velocity_count_1hour        84.15      2.74                    0.0      5.31
rule_velocity_count_24hour        71.52      1.61                    0.0      3.14
rule_channel_high_velocity        81.18      2.10                    0.0      4.10

❌ POOR RULES (36 rules):
   (Precision <40% OR FPR ≥20%) - Consider removing or tuning
                               rule_name  precision_%  recall_% 

In [40]:
# Save rule performance metrics to CSV
rule_perf_output_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rule_performance_metrics.csv"
rule_perf_df.to_csv(rule_perf_output_path, index=False)
print(f"\n✅ Rule performance metrics saved to: {rule_perf_output_path}")

# Summary statistics
print("\n" + "="*100)
print("OVERALL RULE PERFORMANCE STATISTICS")
print("="*100)
print(f"\nPrecision Statistics:")
print(f"  • Mean Precision: {rule_perf_df['precision_%'].mean():.2f}%")
print(f"  • Median Precision: {rule_perf_df['precision_%'].median():.2f}%")
print(f"  • Std Dev: {rule_perf_df['precision_%'].std():.2f}%")
print(f"  • Min: {rule_perf_df['precision_%'].min():.2f}%")
print(f"  • Max: {rule_perf_df['precision_%'].max():.2f}%")

print(f"\nFalse Positive Rate Statistics:")
print(f"  • Mean FPR: {rule_perf_df['false_positive_rate_%'].mean():.2f}%")
print(f"  • Median FPR: {rule_perf_df['false_positive_rate_%'].median():.2f}%")
print(f"  • Std Dev: {rule_perf_df['false_positive_rate_%'].std():.2f}%")
print(f"  • Min: {rule_perf_df['false_positive_rate_%'].min():.2f}%")
print(f"  • Max: {rule_perf_df['false_positive_rate_%'].max():.2f}%")

print(f"\nRecall Statistics:")
print(f"  • Mean Recall: {rule_perf_df['recall_%'].mean():.2f}%")
print(f"  • Median Recall: {rule_perf_df['recall_%'].median():.2f}%")
print(f"  • Std Dev: {rule_perf_df['recall_%'].std():.2f}%")

print(f"\nF1 Score Statistics:")
print(f"  • Mean F1: {rule_perf_df['f1_score'].mean():.2f}")
print(f"  • Median F1: {rule_perf_df['f1_score'].median():.2f}")
print(f"  • Std Dev: {rule_perf_df['f1_score'].std():.2f}")


✅ Rule performance metrics saved to: /root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rule_performance_metrics.csv

OVERALL RULE PERFORMANCE STATISTICS

Precision Statistics:
  • Mean Precision: 12.99%
  • Median Precision: 0.58%
  • Std Dev: 28.06%
  • Min: 0.00%
  • Max: 98.19%

False Positive Rate Statistics:
  • Mean FPR: 10.52%
  • Median FPR: 1.44%
  • Std Dev: 20.55%
  • Min: 0.00%
  • Max: 68.31%

Recall Statistics:
  • Mean Recall: 14.41%
  • Median Recall: 2.46%
  • Std Dev: 24.19%

F1 Score Statistics:
  • Mean F1: 1.64
  • Median F1: 1.00
  • Std Dev: 2.08


## SQL Rule Generation with Thresholds

Convert fraud detection rules to SQL format with proper thresholds for implementation in database queries or real-time scoring systems.

In [41]:
# Generate SQL rules with thresholds
print("🔧 Generating SQL Rules with Thresholds...\n")

sql_rules = []

# Define rule mappings with SQL logic and thresholds
rule_definitions = {
    # VELOCITY RULES
    'rule_velocity_count_15min': {
        'category': 'VELOCITY',
        'description': 'More than 3 transactions in 15 minutes',
        'threshold': 3,
        'sql': """
-- Rule: High transaction velocity in 15 minutes
COUNT(*) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 900 PRECEDING AND CURRENT ROW
) > 3""",
        'clickhouse_sql': """
-- ClickHouse: High transaction velocity in 15 minutes
countIf(
    toUnixTimestamp(trans_initiate_time) >= toUnixTimestamp(trans_initiate_time) - 900
) OVER (PARTITION BY ac_from ORDER BY toUnixTimestamp(trans_initiate_time)) > 3"""
    },
    
    'rule_velocity_count_1hour': {
        'category': 'VELOCITY',
        'description': 'More than 5 transactions in 1 hour',
        'threshold': 5,
        'sql': """
-- Rule: High transaction velocity in 1 hour
COUNT(*) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
) > 5"""
    },
    
    'rule_velocity_count_24hour': {
        'category': 'VELOCITY',
        'description': 'More than 10 transactions in 24 hours',
        'threshold': 10,
        'sql': """
-- Rule: High transaction velocity in 24 hours
COUNT(*) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW
) > 10"""
    },
    
    'rule_velocity_amount_3x_daily': {
        'category': 'VELOCITY',
        'description': 'Transaction amount exceeds 3x daily average in 1 hour',
        'threshold': 3.0,
        'sql': """
-- Rule: Amount velocity exceeds 3x daily average
SUM(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
) > 3 * AVG(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
)"""
    },
    
    'rule_velocity_multiple_channels': {
        'category': 'VELOCITY',
        'description': 'Multiple channels used in 30 minutes',
        'threshold': 2,
        'sql': """
-- Rule: Multiple channels in short time
COUNT(DISTINCT trx_channel) OVER (
    PARTITION BY ac_from 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
) > 1"""
    },
    
    # TIME-BASED RULES
    'rule_time_offpeak_hours': {
        'category': 'TIME',
        'description': 'Transaction during off-peak hours (1 AM - 5 AM)',
        'threshold': None,
        'sql': """
-- Rule: Off-peak hours transaction
HOUR(trans_initiate_time) BETWEEN 1 AND 4"""
    },
    
    'rule_time_weekend_night': {
        'category': 'TIME',
        'description': 'Late night transaction on weekend',
        'threshold': None,
        'sql': """
-- Rule: Weekend night transaction
DAYOFWEEK(trans_initiate_time) IN (1, 7) 
AND (HOUR(trans_initiate_time) >= 23 OR HOUR(trans_initiate_time) < 6)"""
    },
    
    'rule_time_unusual_hour': {
        'category': 'TIME',
        'description': 'Transaction at unusual hour',
        'threshold': None,
        'sql': """
-- Rule: Unusual hour (leverages pre-computed feature)
is_unusual_hour = 1"""
    },
    
    # ACCOUNT AGE RULES
    'rule_age_new_high_amount': {
        'category': 'ACCOUNT_AGE',
        'description': 'New account (<30 days) with high amount (>$500)',
        'threshold': {'age_days': 30, 'amount': 500},
        'sql': """
-- Rule: New account with high amount
DATEDIFF(cutoff_date, mbar_registered_date_time) < 30 
AND trx_amt > 500"""
    },
    
    'rule_age_very_new_any_txn': {
        'category': 'ACCOUNT_AGE',
        'description': 'Very new account (<7 days) any transaction',
        'threshold': {'age_days': 7},
        'sql': """
-- Rule: Very new account
DATEDIFF(cutoff_date, mbar_registered_date_time) < 7"""
    },
    
    'rule_age_immediate_activity': {
        'category': 'ACCOUNT_AGE',
        'description': 'First transaction within 1 hour of registration',
        'threshold': {'hours': 1},
        'sql': """
-- Rule: Immediate activity after registration
TIMESTAMPDIFF(HOUR, mbar_registered_date_time, trans_initiate_time) < 1"""
    },
    
    'rule_age_dormant_30_high_amount': {
        'category': 'ACCOUNT_AGE',
        'description': 'Dormant >30 days then high transaction',
        'threshold': {'dormant_days': 30, 'amount': 1000},
        'sql': """
-- Rule: Dormant account with sudden high transaction
(UNIX_TIMESTAMP(trans_initiate_time) - LAG(UNIX_TIMESTAMP(trans_initiate_time)) OVER (
    PARTITION BY ac_from ORDER BY trans_initiate_time
)) / 86400 > 30
AND trx_amt > 1000"""
    },
    
    # AMOUNT RULES
    'rule_amount_high_value': {
        'category': 'AMOUNT',
        'description': 'High value transaction (>$5000)',
        'threshold': 5000,
        'sql': """
-- Rule: High value transaction
trx_amt > 5000"""
    },
    
    'rule_amount_round_number': {
        'category': 'AMOUNT',
        'description': 'Round number amount (exact multiples of 1000)',
        'threshold': 1000,
        'sql': """
-- Rule: Round number transaction
trx_amt >= 1000 AND MOD(trx_amt, 1000) = 0"""
    },
    
    'rule_amount_structuring': {
        'category': 'AMOUNT',
        'description': 'Just-below-threshold amounts (structuring)',
        'threshold': None,
        'sql': """
-- Rule: Structuring (just below thresholds)
(trx_amt BETWEEN 4900 AND 4999) 
OR (trx_amt BETWEEN 9900 AND 9999)"""
    },
    
    'rule_amount_spike': {
        'category': 'AMOUNT',
        'description': 'Amount >300% of historical average',
        'threshold': 3.0,
        'sql': """
-- Rule: Amount spike
trx_amt > 3 * AVG(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY trans_initiate_time
    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
)"""
    },
    
    'rule_amount_3_stddev': {
        'category': 'AMOUNT',
        'description': 'Amount >3 standard deviations from average',
        'threshold': 3.0,
        'sql': """
-- Rule: Statistical outlier (3 sigma)
trx_amt > AVG(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY trans_initiate_time
    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
) + 3 * STDDEV(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY trans_initiate_time
    ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING
)"""
    },
    
    # CHANNEL RULES
    'rule_channel_pgw_new_high': {
        'category': 'CHANNEL',
        'description': 'Payment Gateway + new account + high amount',
        'threshold': {'age_days': 30, 'amount': 1000},
        'sql': """
-- Rule: Payment Gateway risk pattern
trx_channel IN ('Payment Gateway', 'PGW')
AND DATEDIFF(cutoff_date, mbar_registered_date_time) < 30
AND trx_amt > 1000"""
    },
    
    'rule_channel_mobile_new_high': {
        'category': 'CHANNEL',
        'description': 'Mobile App + new account + high amount',
        'threshold': {'age_days': 30, 'amount': 1000},
        'sql': """
-- Rule: Mobile App risk pattern
trx_channel IN ('Mobile App', 'NEW_JC_APP')
AND DATEDIFF(cutoff_date, mbar_registered_date_time) < 30
AND trx_amt > 1000"""
    },
    
    # BEHAVIORAL RULES
    'rule_behavior_new_recipient_high': {
        'category': 'BEHAVIORAL',
        'description': 'First transaction to new recipient with high amount',
        'threshold': 1000,
        'sql': """
-- Rule: New recipient with high amount
ROW_NUMBER() OVER (
    PARTITION BY ac_from, ac_to 
    ORDER BY trans_initiate_time
) = 1
AND trx_amt > 1000"""
    },
    
    'rule_behavior_amount_jump': {
        'category': 'BEHAVIORAL',
        'description': 'Sudden 5x amount increase',
        'threshold': 5.0,
        'sql': """
-- Rule: Sudden amount jump
trx_amt > 5 * AVG(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY trans_initiate_time
    ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING
)
AND AVG(trx_amt) OVER (
    PARTITION BY ac_from 
    ORDER BY trans_initiate_time
    ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING
) < 1000"""
    },
    
    # RECIPIENT RULES
    'rule_recipient_money_mule': {
        'category': 'RECIPIENT',
        'description': 'Same recipient from >10 senders in 24 hours',
        'threshold': 10,
        'sql': """
-- Rule: Money mule indicator
COUNT(DISTINCT ac_from) OVER (
    PARTITION BY ac_to 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW
) > 10"""
    },
    
    'rule_recipient_high_amount_1hour': {
        'category': 'RECIPIENT',
        'description': 'Recipient receives >$10,000 in 1 hour',
        'threshold': 10000,
        'sql': """
-- Rule: High amount to recipient in short time
SUM(trx_amt) OVER (
    PARTITION BY ac_to 
    ORDER BY UNIX_TIMESTAMP(trans_initiate_time)
    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
) > 10000"""
    },
    
    # FRAUD RING RULES
    'rule_ring_known_fraudster_sender': {
        'category': 'FRAUD_RING',
        'description': 'Transaction from known fraud ring member',
        'threshold': None,
        'sql': """
-- Rule: Known fraudster sender
ac_from IN (
    SELECT ac_from 
    FROM fraud_accounts_blacklist
    WHERE status = 'HIGH_RISK'
)"""
    }
}

print(f"📝 Generated SQL rules for {len(rule_definitions)} fraud detection patterns\n")

🔧 Generating SQL Rules with Thresholds...

📝 Generated SQL rules for 24 fraud detection patterns



In [42]:
# Merge rule performance with SQL definitions
sql_rules_with_performance = []

for rule_name, rule_def in rule_definitions.items():
    # Get performance metrics if available
    perf_data = rule_perf_df[rule_perf_df['rule_name'] == rule_name]
    
    if len(perf_data) > 0:
        perf_row = perf_data.iloc[0]
        sql_rules_with_performance.append({
            'rule_name': rule_name,
            'category': rule_def['category'],
            'description': rule_def['description'],
            'threshold': str(rule_def['threshold']),
            'precision_%': perf_row['precision_%'],
            'recall_%': perf_row['recall_%'],
            'false_positive_rate_%': perf_row['false_positive_rate_%'],
            'f1_score': perf_row['f1_score'],
            'lift': perf_row['lift'],
            'times_triggered': perf_row['times_triggered'],
            'sql_logic': rule_def['sql']
        })
    else:
        sql_rules_with_performance.append({
            'rule_name': rule_name,
            'category': rule_def['category'],
            'description': rule_def['description'],
            'threshold': str(rule_def['threshold']),
            'precision_%': 0,
            'recall_%': 0,
            'false_positive_rate_%': 0,
            'f1_score': 0,
            'lift': 0,
            'times_triggered': 0,
            'sql_logic': rule_def['sql']
        })

sql_rules_df = pd.DataFrame(sql_rules_with_performance)

print("✅ Merged performance metrics with SQL rule definitions")
print(f"   Total rules with SQL: {len(sql_rules_df)}\n")

✅ Merged performance metrics with SQL rule definitions
   Total rules with SQL: 24



In [ ]:
# Display SQL rules sorted by performance
print("="*100)
print("FRAUD DETECTION RULES - SQL FORMAT (Sorted by Precision)")
print("="*100)

# Sort by precision
sql_rules_sorted = sql_rules_df.sort_values('precision_%', ascending=False)

for idx, row in sql_rules_sorted.iterrows():
    print(f"\n{'='*100}")
    print(f"Rule: {row['rule_name']}")
    print(f"Category: {row['category']}")
    print(f"Description: {row['description']}")
    print(f"Threshold: {row['threshold']}")
    print(f"\nPerformance Metrics:")
    print(f"  • Precision: {row['precision_%']:.2f}%")
    print(f"  • Recall: {row['recall_%']:.2f}%")
    print(f"  • False Positive Rate: {row['false_positive_rate_%']:.2f}%")
    print(f"  • F1 Score: {row['f1_score']:.2f}")
    print(f"  • Lift: {row['lift']:.2f}x")
    print(f"  • Times Triggered: {row['times_triggered']:,}")
    print(f"\nSQL Logic:")
    print(row['sql_logic'])
    print(f"{'='*100}")

print(f"\n\n✅ Displayed {len(sql_rules_sorted)} SQL rules sorted by precision")

In [ ]:
# Save SQL rules to files
print("\n💾 Saving SQL Rules to Files...\n")

# 1. Save complete SQL rules with performance
sql_rules_csv_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_sql_with_performance.csv"
sql_rules_df.to_csv(sql_rules_csv_path, index=False)
print(f"✅ SQL rules with performance saved to: {sql_rules_csv_path}")

# 2. Generate SQL script file for database implementation
sql_script_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_detection_rules.sql"

with open(sql_script_path, 'w') as f:
    f.write("-- =====================================================\n")
    f.write("-- FRAUD DETECTION RULES - SQL IMPLEMENTATION\n")
    f.write("-- =====================================================\n")
    f.write("-- Generated: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n")
    f.write("-- Total Rules: " + str(len(sql_rules_df)) + "\n")
    f.write("-- =====================================================\n\n")
    
    # Group rules by category
    for category in sql_rules_df['category'].unique():
        f.write(f"\n-- =====================================================\n")
        f.write(f"-- CATEGORY: {category}\n")
        f.write(f"-- =====================================================\n\n")
        
        category_rules = sql_rules_df[sql_rules_df['category'] == category].sort_values('precision_%', ascending=False)
        
        for idx, row in category_rules.iterrows():
            f.write(f"-- Rule: {row['rule_name']}\n")
            f.write(f"-- Description: {row['description']}\n")
            f.write(f"-- Threshold: {row['threshold']}\n")
            f.write(f"-- Performance: Precision={row['precision_%']:.2f}%, ")
            f.write(f"FPR={row['false_positive_rate_%']:.2f}%, ")
            f.write(f"Recall={row['recall_%']:.2f}%\n")
            f.write(row['sql_logic'])
            f.write("\n\n")

print(f"✅ SQL script saved to: {sql_script_path}")

# 3. Generate ClickHouse-specific SQL
clickhouse_sql_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_detection_rules_clickhouse.sql"

with open(clickhouse_sql_path, 'w') as f:
    f.write("-- =====================================================\n")
    f.write("-- FRAUD DETECTION RULES - CLICKHOUSE IMPLEMENTATION\n")
    f.write("-- =====================================================\n")
    f.write("-- Generated: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n")
    f.write("-- Note: ClickHouse-specific syntax and window functions\n")
    f.write("-- =====================================================\n\n")
    
    f.write("""
-- Example: Complete fraud scoring query for ClickHouse
SELECT 
    trans_id,
    ac_from,
    ac_to,
    cutoff_date,
    trans_initiate_time,
    trx_channel,
    trx_type,
    trx_amt,
    fraud_flag,
    
    -- VELOCITY RULES (weight: 3.0)
    IF(
        countIf(toUnixTimestamp(trans_initiate_time) >= toUnixTimestamp(trans_initiate_time) - 900) 
        OVER (PARTITION BY ac_from ORDER BY toUnixTimestamp(trans_initiate_time) 
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) > 3,
        1, 0
    ) AS rule_velocity_count_15min,
    
    IF(
        countIf(toUnixTimestamp(trans_initiate_time) >= toUnixTimestamp(trans_initiate_time) - 3600) 
        OVER (PARTITION BY ac_from ORDER BY toUnixTimestamp(trans_initiate_time) 
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) > 5,
        1, 0
    ) AS rule_velocity_count_1hour,
    
    -- TIME RULES (weight: 1.5)
    IF(toHour(trans_initiate_time) BETWEEN 1 AND 4, 1, 0) AS rule_time_offpeak_hours,
    
    IF(
        toDayOfWeek(trans_initiate_time) IN (6, 7) 
        AND (toHour(trans_initiate_time) >= 23 OR toHour(trans_initiate_time) < 6),
        1, 0
    ) AS rule_time_weekend_night,
    
    -- ACCOUNT AGE RULES (weight: 2.0)
    IF(
        dateDiff('day', mbar_registered_date_time, cutoff_date) < 30 
        AND trx_amt > 500,
        1, 0
    ) AS rule_age_new_high_amount,
    
    IF(dateDiff('day', mbar_registered_date_time, cutoff_date) < 7, 1, 0) AS rule_age_very_new_any_txn,
    
    -- AMOUNT RULES (weight: 2.5)
    IF(trx_amt > 5000, 1, 0) AS rule_amount_high_value,
    
    IF(trx_amt >= 1000 AND (trx_amt % 1000) = 0, 1, 0) AS rule_amount_round_number,
    
    IF(
        (trx_amt BETWEEN 4900 AND 4999) OR (trx_amt BETWEEN 9900 AND 9999),
        1, 0
    ) AS rule_amount_structuring,
    
    -- CHANNEL RULES (weight: 1.5)
    IF(
        trx_channel IN ('Payment Gateway', 'PGW')
        AND dateDiff('day', mbar_registered_date_time, cutoff_date) < 30
        AND trx_amt > 1000,
        1, 0
    ) AS rule_channel_pgw_new_high,
    
    -- Calculate weighted risk score
    (
        -- Velocity rules * 3.0
        (IF(countIf(toUnixTimestamp(trans_initiate_time) >= toUnixTimestamp(trans_initiate_time) - 900) 
             OVER (PARTITION BY ac_from ORDER BY toUnixTimestamp(trans_initiate_time)) > 3, 1, 0) * 3.0) +
        
        -- Time rules * 1.5
        (IF(toHour(trans_initiate_time) BETWEEN 1 AND 4, 1, 0) * 1.5) +
        
        -- Account age rules * 2.0
        (IF(dateDiff('day', mbar_registered_date_time, cutoff_date) < 30 AND trx_amt > 500, 1, 0) * 2.0) +
        
        -- Amount rules * 2.5
        (IF(trx_amt > 5000, 1, 0) * 2.5) +
        (IF(trx_amt >= 1000 AND (trx_amt % 1000) = 0, 1, 0) * 2.5)
        
    ) AS fraud_risk_score,
    
    -- Risk category
    CASE
        WHEN fraud_risk_score >= 20 THEN 'CRITICAL'
        WHEN fraud_risk_score >= 15 THEN 'HIGH'
        WHEN fraud_risk_score >= 10 THEN 'MEDIUM'
        WHEN fraud_risk_score >= 5 THEN 'LOW'
        ELSE 'MINIMAL'
    END AS risk_category

FROM stixor_fraud_features_distributed
WHERE cutoff_date BETWEEN '2025-05-01' AND '2025-06-30'
    AND mbar_account_type_name = 'Customer Account'
ORDER BY fraud_risk_score DESC
LIMIT 1000;
""")

print(f"✅ ClickHouse SQL script saved to: {clickhouse_sql_path}")

# 4. Generate Python scoring function
python_rules_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_detection_rules.py"

with open(python_rules_path, 'w') as f:
    f.write('"""Fraud Detection Rules - Python Implementation"""\n\n')
    f.write('from datetime import datetime, timedelta\n\n')
    f.write('def calculate_fraud_risk_score(transaction, transaction_history):\n')
    f.write('    """\n')
    f.write('    Calculate fraud risk score for a transaction.\n')
    f.write('    \n')
    f.write('    Args:\n')
    f.write('        transaction: Dict with current transaction data\n')
    f.write('        transaction_history: List of previous transactions for the account\n')
    f.write('    \n')
    f.write('    Returns:\n')
    f.write('        tuple: (risk_score, risk_category, triggered_rules)\n')
    f.write('    """\n')
    f.write('    score = 0\n')
    f.write('    triggered_rules = []\n\n')
    f.write('    # Implement rules here based on SQL logic\n')
    f.write('    # TODO: Add rule implementations\n\n')
    f.write('    return score, risk_category, triggered_rules\n')

print(f"✅ Python scoring function template saved to: {python_rules_path}")

print("\n" + "="*100)
print("SUMMARY: SQL RULES EXPORT")
print("="*100)
print(f"✅ Files created:")
print(f"   1. {sql_rules_csv_path}")
print(f"      - Complete rules with performance metrics (CSV)")
print(f"   2. {sql_script_path}")
print(f"      - Generic SQL rules implementation")
print(f"   3. {clickhouse_sql_path}")
print(f"      - ClickHouse-specific SQL with example query")
print(f"   4. {python_rules_path}")
print(f"      - Python function template for real-time scoring")
print("="*100)

In [ ]:
# Generate README for SQL rules
readme_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/FRAUD_RULES_README.md"

with open(readme_path, 'w') as f:
    f.write("# Fraud Detection Rules - Implementation Guide\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("## Overview\n\n")
    f.write(f"This directory contains {len(sql_rules_df)} fraud detection rules with SQL implementations ")
    f.write("and performance metrics.\n\n")
    
    f.write("## Files\n\n")
    f.write("1. **fraud_rule_performance_metrics.csv**\n")
    f.write("   - Complete performance metrics for all rules\n")
    f.write("   - Includes precision, recall, FPR, F1 score, lift\n\n")
    
    f.write("2. **fraud_rules_sql_with_performance.csv**\n")
    f.write("   - SQL rules with performance metrics\n")
    f.write("   - Includes thresholds and SQL logic\n\n")
    
    f.write("3. **fraud_detection_rules.sql**\n")
    f.write("   - Generic SQL implementation\n")
    f.write("   - Compatible with MySQL, PostgreSQL, etc.\n\n")
    
    f.write("4. **fraud_detection_rules_clickhouse.sql**\n")
    f.write("   - ClickHouse-specific implementation\n")
    f.write("   - Optimized for ClickHouse window functions\n\n")
    
    f.write("5. **fraud_detection_rules.py**\n")
    f.write("   - Python function template\n")
    f.write("   - For real-time scoring API\n\n")
    
    f.write("## Rule Categories\n\n")
    
    for category in sql_rules_df['category'].unique():
        category_rules = sql_rules_df[sql_rules_df['category'] == category]
        f.write(f"### {category} ({len(category_rules)} rules)\n\n")
        
        avg_precision = category_rules['precision_%'].mean()
        avg_fpr = category_rules['false_positive_rate_%'].mean()
        
        f.write(f"- Average Precision: {avg_precision:.2f}%\n")
        f.write(f"- Average FPR: {avg_fpr:.2f}%\n")
        f.write(f"- Rules:\n")
        
        for _, rule in category_rules.sort_values('precision_%', ascending=False).iterrows():
            f.write(f"  - `{rule['rule_name']}`: {rule['description']}\n")
            f.write(f"    - Precision: {rule['precision_%']:.2f}%, ")
            f.write(f"FPR: {rule['false_positive_rate_%']:.2f}%\n")
        
        f.write("\n")
    
    f.write("## Performance Summary\n\n")
    
    # Top rules
    top_10 = sql_rules_df.sort_values('precision_%', ascending=False).head(10)
    f.write("### Top 10 Rules by Precision\n\n")
    f.write("| Rank | Rule | Precision | FPR | Recall | F1 |\n")
    f.write("|------|------|-----------|-----|--------|----|\n")
    
    for idx, (rank, row) in enumerate(top_10.iterrows(), 1):
        f.write(f"| {idx} | {row['rule_name']} | ")
        f.write(f"{row['precision_%']:.2f}% | {row['false_positive_rate_%']:.2f}% | ")
        f.write(f"{row['recall_%']:.2f}% | {row['f1_score']:.2f} |\n")
    
    f.write("\n## Implementation Notes\n\n")
    f.write("### Recommended Rule Weights\n\n")
    f.write("- **FRAUD_RING**: 5.0 (highest priority - known fraudsters)\n")
    f.write("- **VELOCITY**: 3.0 (strong fraud indicator)\n")
    f.write("- **AMOUNT**: 2.5 (high-value transactions)\n")
    f.write("- **RECIPIENT**: 2.5 (money mule patterns)\n")
    f.write("- **ACCOUNT_AGE**: 2.0 (new account risks)\n")
    f.write("- **BEHAVIORAL**: 2.0 (anomaly detection)\n")
    f.write("- **TIME**: 1.5 (temporal patterns)\n")
    f.write("- **CHANNEL**: 1.5 (channel-specific risks)\n\n")
    
    f.write("### Risk Score Thresholds\n\n")
    f.write("- **CRITICAL**: ≥ 20 points (immediate review required)\n")
    f.write("- **HIGH**: ≥ 15 points (priority investigation)\n")
    f.write("- **MEDIUM**: ≥ 10 points (automated review)\n")
    f.write("- **LOW**: ≥ 5 points (log for analysis)\n")
    f.write("- **MINIMAL**: < 5 points (normal transaction)\n\n")
    
    f.write("### Rule Tuning Guidelines\n\n")
    f.write("1. **High Precision, Low FPR**: Deploy immediately in production\n")
    f.write("2. **Good Precision, Moderate FPR**: Use with manual review\n")
    f.write("3. **Low Precision, High FPR**: Consider removing or adjusting thresholds\n\n")
    
    f.write("### Real-Time Implementation\n\n")
    f.write("For real-time scoring:\n")
    f.write("1. Pre-compute historical features (velocity, patterns)\n")
    f.write("2. Cache account risk profiles (Redis/Memcached)\n")
    f.write("3. Maintain fraud account blacklist\n")
    f.write("4. Use async processing for non-blocking rules\n")
    f.write("5. Implement circuit breakers for performance\n\n")
    
    f.write("## Contact\n\n")
    f.write("For questions or improvements, contact the fraud analytics team.\n")

print(f"✅ README saved to: {readme_path}")

print("\n" + "="*100)
print("🎉 FRAUD RULES ANALYSIS AND SQL EXPORT COMPLETE!")
print("="*100)
print(f"\n📊 Summary:")
print(f"  • Total rules analyzed: {len(rule_perf_df)}")
print(f"  • Rules with SQL implementation: {len(sql_rules_df)}")
print(f"  • Average precision: {rule_perf_df['precision_%'].mean():.2f}%")
print(f"  • Average FPR: {rule_perf_df['false_positive_rate_%'].mean():.2f}%")
print(f"  • Files generated: 6")
print(f"\n📁 Output directory: /root/research-dir/dev/jazzcash-fraud-detection/data/")
print(f"\n🚀 Next Steps:")
print(f"  1. Review rule performance metrics CSV")
print(f"  2. Implement high-precision rules first")
print(f"  3. Tune thresholds for poor-performing rules")
print(f"  4. Deploy ClickHouse SQL for real-time scoring")
print(f"  5. Build API using Python template")
print("="*100)

## Test Fraud Rules on July 2025 Data (Out-of-Sample Validation)

Load fresh data from July 2025 to validate rule performance on unseen transactions. This provides an unbiased assessment of rule effectiveness.

In [ ]:
from pyspark.sql import SparkSession

jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]

fraud_channels =  ['API', 'ATM', 'BIO', 'BISP', 'Business App', 'Business App API', 'Cheetay', 'JC Keyboard', 'Merchant Payment', 'Mobile App', 'NEW_JC_APP', 'PAYPAK', 'PGW', 'Payment Gateway', 'QR Payment', 'Self Care App', 'THIRD_PARTY_WEB', 'USSD', 'USSD API', 'USSD_API', 'VRG']
fraud_types =  ['Online Payment', 'PTS ATM Withdrawal', 'PTS Purchase Payment', 'Transfer(C2C)', 'MFS Card Withdraw', 'Cash in', 'Cash out', 'IBFT Outgoing Customer', 'IBFT Outgoing OTC', 'Merchant Payment', 'Transfer(B2C)', 'Transfer(C2B)', 'Jazz Load (Prepaid top-up)', 'Business Cash Out', 'Customer Remit To CNIC', 'Donation', 'Get Loan', 'IBFT Credit', 'Others', 'Utility Bills Payment', 'Purchase Payment', 'Auto Debit', 'Indigo Bills (Postpaid payment)', '']

channels_in_clause = ", ".join([f"'{ch}'" for ch in fraud_channels])
types_in_clause = ", ".join([f"'{tp}'" for tp in fraud_types])
# selected_cols = ['trans_id', 'ac_from', 'ac_to', 'data_date', 'trans_initiate_time', 'cutoff_date', 'fraud_flag', 'trx_channel', 'trx_type', 'start_balance', 'end_balance', 'trx_amt', 'mbar_registered_date_time', 'mbar_a_c_status',  'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'is_business_hours', 'is_unusual_hour', 'night_weekend_combo', 'start_balance_log', 'balance_change', 'balance_change_pct', 'txn_txns_3d', 'txn_total_amount_3d', 'txn_avg_amount_3d', 'txn_max_amount_3d', 'txn_min_amount_3d', 'txn_unique_recipients_3d', 'txn_unique_channels_3d', 'txn_unique_types_3d', 'txn_is_high_activity_3d', 'txn_multi_channel_recent', 'txn_amount_deviation_from_avg', 'txn_night_txns_3d', 'txn_weekend_txns_3d', 'channel_new_jc_app', 'channel_ussd', 'channel_ussd_api', 'channel_payment_gateway', 'channel_mobile_app', 'type_transfer_c2c', 'type_transfer_c2b', 'type_bill_payment', 'type_mobile_load', 'user_total_txns_3d', 'user_total_amount_3d', 'user_avg_amount_3d', 'user_median_amount_3d', 'user_max_amount_3d', 'user_min_amount_3d', 'user_unique_recipients_3d', 'user_unique_channels_3d', 'user_unique_types_3d', 'user_total_txns_7d', 'user_total_amount_7d', 'user_avg_amount_7d', 'user_median_amount_7d', 'user_max_amount_7d', 'user_min_amount_7d', 'user_unique_recipients_7d', 'user_unique_channels_7d', 'user_unique_types_7d', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_channel_diversity_score_7d', 'user_most_used_type_7d', 'user_last_used_type', 'user_type_diversity_score_7d', 'user_night_txns_7d', 'user_weekend_txns_7d', 'user_peak_hour_txns_7d', 'user_off_peak_hour_txns_7d', 'user_avg_start_balance_7d', 'user_avg_end_balance_7d', 'user_min_balance_7d', 'user_max_balance_7d', 'user_balance_volatility_7d', 'user_top_recipient_7d', 'user_avg_amount_per_recipient_7d', 'user_max_amount_to_single_recipient_7d', 'user_recipient_concentration_ratio_7d', 'user_avg_time_between_txns_7d', 'user_txn_frequency_score_7d', 'user_first_txn_time', 'user_last_txn_time', 'user_days_since_last_txn']

CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

# spark.stop()
# Initialize Spark session with JARs
spark = SparkSession.builder \
    .appName("data_loading") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "150g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()

# CORRECT URL: Use HTTP port 8123 (not native port 9000)
url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user'] 
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

In [2]:
# Load July 2025 data for testing
print("📅 Loading July 2025 data for rule validation...\n")

test_start_date = '2025-07-01'
test_end_date = '2025-07-30'

# Load all fraud transactions from July
fraud_test_query = f"""
SELECT *
FROM stixor_fraud_features_distributed
WHERE cutoff_date BETWEEN '{test_start_date}' AND '{test_end_date}'
"""

fraud_test_subquery = f"({fraud_test_query}) AS fraud_test_data"

print(f"🚨 Loading all fraud transactions from July 2025...")
df = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', fraud_test_subquery)
    .option('fetchsize', '1000000')
    .option("partitionColumn", "cutoff_date")
    .option('lowerBound', test_start_date)
    .option('upperBound', test_end_date)
    .option('numPartitions', '10')
    .load())


📅 Loading July 2025 data for rule validation...

🚨 Loading all fraud transactions from July 2025...


In [3]:
df.count()

391723456

In [ ]:
# Load all fraud transactions from July
rule_1_query = f"""
SELECT trans_id, fraud_flag,case when DATEDIFF('day',toDate(mbar_registered_date_time) ,cutoff_date) < 30 AND trx_amt>10000 then 1 else 0 end rule_1  from public.stixor_fraud_features_distributed where cutoff_date between '2025-07-01' and '2025-07-30'
"""
rule_1_query = f"({rule_1_query}) AS fraud_test_data"

rule_1_df = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', rule_1_query)
    .option('numPartitions', '30')
    .load())


In [ ]:
rule_1_df.show(5)

+-----------+----------+------+
|   trans_id|fraud_flag|rule_1|
+-----------+----------+------+
|84395861061|         0|     0|
|84395864604|         0|     0|
|84395865612|         0|     0|
|84395865655|         0|     0|
|84395865659|         0|     0|
+-----------+----------+------+
only showing top 5 rows


In [ ]:
# SELECT 
#     'Rule 1: High Amount' AS rule_name,
    
#     countIf(rule_1 = 1 AND fraud_flag = 1) AS true_positives,
#     countIf(rule_1 = 1 AND fraud_flag = 0) AS false_positives,
#     countIf(rule_1 = 0 AND fraud_flag = 1) AS false_negatives,
#     countIf(rule_1 = 0 AND fraud_flag = 0) AS true_negatives,
    
#     round(countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1 = 1), 0), 2) AS precision,
#     round(countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 2) AS recall,
#     round(2 * (countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1 = 1), 0)) * 
#               (countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)) / 
#           nullIf((countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1 = 1), 0)) + 
#                  (countIf(rule_1 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)), 0), 2) AS f1_score
# FROM
# (SELECT trans_id, fraud_flag,case when DATEDIFF('day',toDate(mbar_registered_date_time) ,cutoff_date) < 30 AND trx_amt>10000 then 1 else 0 end rule_1  from public.stixor_fraud_features_distributed where cutoff_date between '2025-07-01' and '2025-07-30')


In [ ]:
# Load all fraud transactions from July
rule_2_query = f"""
SELECT trans_id, fraud_flag,case when DATEDIFF('hour',toDate(mbar_registered_date_time) ,cutoff_date) < 1 AND trx_amt>10000 then 1 else 0 end rule_2  from public.stixor_fraud_features_distributed where cutoff_date between '2025-07-01' and '2025-07-30'
"""
rule_2_query = f"({rule_2_query}) AS fraud_test_data"

rule_2_df = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', rule_2_query)
    .option('numPartitions', '30')
    .load())

rule_2_df.show(5)

+-----------+----------+------+
|   trans_id|fraud_flag|rule_2|
+-----------+----------+------+
|84317504152|         0|     0|
|84384848261|         0|     0|
|84810977524|         0|     0|
|84951019113|         0|     0|
|83947708235|         0|     0|
+-----------+----------+------+
only showing top 5 rows


In [ ]:
# SELECT 
#     "Rule 3: High Velocity" AS rule_name,
#     countIf(rule_3 = 1 AND fraud_flag = 1) AS true_positives,
#     countIf(rule_3 = 1 AND fraud_flag = 0) AS false_positives,
#     countIf(rule_3 = 0 AND fraud_flag = 1) AS false_negatives,
#     countIf(rule_3 = 0 AND fraud_flag = 0) AS true_negatives,
#     round(countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_3 = 1), 0), 2) AS precision,
#     round(countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 2) AS recall,
#     round(2 * (countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_3 = 1), 0)) * 
#               (countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)) /
#           nullIf((countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_3 = 1), 0)) + 
#                  (countIf(rule_3 = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)), 0), 2) AS f1_score
# FROM (
# SELECT trans_id,
#     fraud_flag,
#     CASE 
#         WHEN COUNT(*) OVER (
#             PARTITION BY ac_from 
#             ORDER BY toUnixTimestamp(trans_initiate_time) ASC 
#             RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
#         ) > 10 
#         THEN 1 
#         ELSE 0 
#     END AS rule_3
# FROM public.stixor_fraud_features_distributed
# WHERE cutoff_date BETWEEN '2025-07-01' AND '2025-07-30' AND mbar_account_type_name = 'Customer Account' )

In [ ]:
# Load all fraud transactions from July
rule_3_query = f"""
SELECT 
    trans_id,
    fraud_flag,
    CASE 
        WHEN COUNT(*) OVER (
            PARTITION BY ac_from 
            ORDER BY toUnixTimestamp(trans_initiate_time) ASC 
            RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
        ) > 10 
        THEN 1 
        ELSE 0 
    END AS rule_3
FROM public.stixor_fraud_features_distributed
WHERE cutoff_date BETWEEN '2025-07-01' AND '2025-07-30'
    AND mbar_account_type_name = 'Customer Account'"""
rule_3_query = f"({rule_3_query}) AS fraud_test_data"

rule_3_df = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', rule_3_query)
    .option('numPartitions', '30')
    .load())

rule_3_df.show(5)

+-----------+----------+------+
|   trans_id|fraud_flag|rule_3|
+-----------+----------+------+
|83868645876|         0|     0|
|83872842911|         0|     0|
|83923579718|         0|     0|
|83923683486|         0|     0|
|83923894076|         0|     0|
+-----------+----------+------+
only showing top 5 rows


In [7]:
from pyspark.sql.functions import unix_timestamp, col, when, count, sum as spark_sum, avg, stddev, size, collect_set, countDistinct, lag, row_number, datediff, lit, coalesce, expr
from pyspark.sql.window import Window

# Apply ALL fraud detection rules to July 2025 test data
print("🔧 Applying fraud detection rules to July 2025 test data...\n")
print("This will apply all 43+ rules across 9 categories\n")
# Start with test data
test_rules_df = test_df

# Add timestamp column
test_rules_df = test_rules_df.withColumn('trans_timestamp', 
    unix_timestamp(col('trans_initiate_time')))

print("✅ Step 1/9: Applying VELOCITY RULES...")
# VELOCITY RULES - Same logic as training
window_15min = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-900, 0)
window_1hour = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-3600, 0)
window_24hour = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-86400, 0)
window_7days = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-604800, 0)

test_rules_df = test_rules_df.withColumn('txn_count_15min', count('*').over(window_15min)) \
    .withColumn('txn_count_1hour', count('*').over(window_1hour)) \
    .withColumn('txn_count_24hour', count('*').over(window_24hour)) \
    .withColumn('txn_count_7days', count('*').over(window_7days))

test_rules_df = test_rules_df.withColumn('rule_velocity_count_15min', 
    when(col('txn_count_15min') > 3, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_1hour', 
    when(col('txn_count_1hour') > 5, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_24hour', 
    when(col('txn_count_24hour') > 10, 1).otherwise(0)) \
    .withColumn('rule_velocity_count_7days_new_account', 
    when((col('txn_count_7days') > 20) & (datediff(col('cutoff_date'), col('mbar_registered_date_time')) < 30), 1).otherwise(0))

# Amount velocity
window_account_history = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-30, -1)
test_rules_df = test_rules_df.withColumn('amount_sum_1hour', spark_sum('trx_amt').over(window_1hour)) \
    .withColumn('amount_sum_15min', spark_sum('trx_amt').over(window_15min)) \
    .withColumn('historical_daily_avg', coalesce(avg('trx_amt').over(window_account_history), lit(0)))

test_rules_df = test_rules_df.withColumn('rule_velocity_amount_3x_daily', 
    when((col('amount_sum_1hour') > col('historical_daily_avg') * 3) & (col('historical_daily_avg') > 0), 1).otherwise(0)) \
    .withColumn('rule_velocity_amount_15min_threshold', 
    when((col('amount_sum_15min') > 10000) & (col('historical_daily_avg') < 2000), 1).otherwise(0))

# Cross-feature velocity
window_30min_channels = Window.partitionBy('ac_from').orderBy('trans_timestamp').rangeBetween(-1800, 0)
window_recipient_24hour = Window.partitionBy('ac_to').orderBy('trans_timestamp').rangeBetween(-86400, 0)

test_rules_df = test_rules_df.withColumn('channels_used_30min', 
    size(collect_set('trx_channel').over(window_30min_channels))) \
    .withColumn('rule_velocity_multiple_channels', 
    when(col('channels_used_30min') > 1, 1).otherwise(0))

# Use collect_set and size instead of countDistinct for window function
test_rules_df = test_rules_df.withColumn('unique_senders_24hour', 
    size(collect_set('ac_from').over(window_recipient_24hour))) \
    .withColumn('rule_velocity_multiple_senders_recipient', 
    when(col('unique_senders_24hour') > 5, 1).otherwise(0))

velocity_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_velocity_')]
test_rules_df = test_rules_df.withColumn('velocity_rules_triggered', 
    sum([col(c) for c in velocity_rule_cols]))

print(f"   ✓ Applied {len(velocity_rule_cols)} velocity rules")

print("\n✅ Step 2/9: Applying TIME-BASED RULES...")
# TIME RULES
test_rules_df = test_rules_df.withColumn('rule_time_offpeak_hours', 
    when((col('hour_of_day') >= 1) & (col('hour_of_day') < 5), 1).otherwise(0)) \
    .withColumn('rule_time_weekend_night', 
    when((col('is_weekend') == 1) & (((col('hour_of_day') >= 23) | (col('hour_of_day') < 6))), 1).otherwise(0)) \
    .withColumn('rule_time_night_weekend_combo', 
    when(col('night_weekend_combo') == 1, 1).otherwise(0))

window_account_all = Window.partitionBy('ac_from').orderBy('trans_timestamp')
test_rules_df = test_rules_df.withColumn('is_first_transaction', 
    when(row_number().over(window_account_all) == 1, 1).otherwise(0)) \
    .withColumn('rule_time_first_txn_unusual', 
    when((col('is_first_transaction') == 1) & (col('is_night') == 1), 1).otherwise(0)) \
    .withColumn('rule_time_unusual_hour', 
    when(col('is_unusual_hour') == 1, 1).otherwise(0))

time_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_time_')]
test_rules_df = test_rules_df.withColumn('time_rules_triggered', 
    sum([col(c) for c in time_rule_cols]))

print(f"   ✓ Applied {len(time_rule_cols)} time-based rules")

print("\n✅ Step 3/9: Applying ACCOUNT AGE RULES...")
# ACCOUNT AGE RULES
test_rules_df = test_rules_df.withColumn('account_age_days', 
    datediff(col('cutoff_date'), col('mbar_registered_date_time')))

test_rules_df = test_rules_df.withColumn('rule_age_new_high_amount', 
    when((col('account_age_days') < 30) & (col('trx_amt') > 500), 1).otherwise(0)) \
    .withColumn('rule_age_very_new_any_txn', 
    when(col('account_age_days') < 7, 1).otherwise(0)) \
    .withColumn('rule_age_fresh_multiple_txns', 
    when((col('account_age_days') < 3) & (col('txn_count_24hour') > 1), 1).otherwise(0)) \
    .withColumn('rule_age_new_high_velocity', 
    when((col('account_age_days') < 30) & (col('txn_count_24hour') > 5), 1).otherwise(0))

test_rules_df = test_rules_df.withColumn('registration_hour', 
    expr("hour(mbar_registered_date_time)")) \
    .withColumn('rule_age_unusual_reg_time', 
    when((col('registration_hour') >= 0) & (col('registration_hour') < 5), 1).otherwise(0))

test_rules_df = test_rules_df.withColumn('time_since_registration_hours', 
    (unix_timestamp(col('trans_initiate_time')) - unix_timestamp(col('mbar_registered_date_time'))) / 3600) \
    .withColumn('rule_age_immediate_activity', 
    when((col('time_since_registration_hours') >= 0) & (col('time_since_registration_hours') < 1), 1).otherwise(0))

window_prev_txn = Window.partitionBy('ac_from').orderBy('trans_timestamp')
test_rules_df = test_rules_df.withColumn('prev_txn_timestamp', 
    lag('trans_timestamp', 1).over(window_prev_txn)) \
    .withColumn('days_since_last_txn', 
    (col('trans_timestamp') - col('prev_txn_timestamp')) / 86400) \
    .withColumn('rule_age_dormant_30_high_amount', 
    when((col('days_since_last_txn') > 30) & (col('trx_amt') > 1000), 1).otherwise(0)) \
    .withColumn('rule_age_dormant_60_rapid', 
    when((col('days_since_last_txn') > 60) & (col('txn_count_1hour') > 2), 1).otherwise(0))

age_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_age_')]
test_rules_df = test_rules_df.withColumn('age_rules_triggered', 
    sum([col(c) for c in age_rule_cols]))

print(f"   ✓ Applied {len(age_rule_cols)} account age rules")

print("\n✅ Step 4/9: Applying AMOUNT-BASED RULES...")
# AMOUNT RULES
window_account_stats = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-100, -1)

test_rules_df = test_rules_df.withColumn('historical_avg_amount', 
    coalesce(avg('trx_amt').over(window_account_stats), lit(0))) \
    .withColumn('historical_stddev_amount', 
    coalesce(stddev('trx_amt').over(window_account_stats), lit(0)))

test_rules_df = test_rules_df.withColumn('rule_amount_95_percentile', 
    when((col('trx_amt') > col('historical_avg_amount') + 2 * col('historical_stddev_amount')) & 
         (col('historical_stddev_amount') > 0), 1).otherwise(0)) \
    .withColumn('rule_amount_3_stddev', 
    when((col('trx_amt') > col('historical_avg_amount') + 3 * col('historical_stddev_amount')) & 
         (col('historical_stddev_amount') > 0), 1).otherwise(0)) \
    .withColumn('rule_amount_high_value', 
    when(col('trx_amt') > 5000, 1).otherwise(0)) \
    .withColumn('rule_amount_round_number', 
    when((col('trx_amt') % 1000 == 0) & (col('trx_amt') >= 1000), 1).otherwise(0))

window_last_5_txns = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-5, -1)
test_rules_df = test_rules_df.withColumn('stddev_last_5_amounts', 
    coalesce(stddev('trx_amt').over(window_last_5_txns), lit(999999))) \
    .withColumn('rule_amount_uniform_pattern', 
    when((col('stddev_last_5_amounts') < 10) & (col('stddev_last_5_amounts') > 0), 1).otherwise(0)) \
    .withColumn('rule_amount_structuring', 
    when(((col('trx_amt') >= 4900) & (col('trx_amt') < 5000)) | 
         ((col('trx_amt') >= 9900) & (col('trx_amt') < 10000)), 1).otherwise(0)) \
    .withColumn('rule_amount_spike', 
    when((col('trx_amt') > col('historical_avg_amount') * 3) & 
         (col('historical_avg_amount') > 0), 1).otherwise(0))

amount_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_amount_')]
test_rules_df = test_rules_df.withColumn('amount_rules_triggered', 
    sum([col(c) for c in amount_rule_cols]))

print(f"   ✓ Applied {len(amount_rule_cols)} amount-based rules")

print("\n✅ Step 5/9: Applying CHANNEL-BASED RULES...")
# CHANNEL RULES
test_rules_df = test_rules_df.withColumn('rule_channel_pgw_new_high', 
    when((col('trx_channel').isin(['Payment Gateway', 'PGW'])) & 
         (col('account_age_days') < 30) & 
         (col('trx_amt') > 1000), 1).otherwise(0)) \
    .withColumn('rule_channel_mobile_new_high', 
    when((col('trx_channel').isin(['Mobile App', 'NEW_JC_APP'])) & 
         (col('account_age_days') < 30) & 
         (col('trx_amt') > 1000), 1).otherwise(0))

if 'user_most_used_channel_7d' in test_rules_df.columns:
    test_rules_df = test_rules_df.withColumn('rule_channel_switch', 
        when((col('trx_channel') != col('user_most_used_channel_7d')) & 
             (col('user_most_used_channel_7d').isNotNull()), 1).otherwise(0))
else:
    test_rules_df = test_rules_df.withColumn('rule_channel_switch', lit(0))

test_rules_df = test_rules_df.withColumn('rule_channel_high_velocity', 
    when(col('txn_count_15min') > 5, 1).otherwise(0))

channel_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_channel_')]
test_rules_df = test_rules_df.withColumn('channel_rules_triggered', 
    sum([col(c) for c in channel_rule_cols]))

print(f"   ✓ Applied {len(channel_rule_cols)} channel-based rules")

print("\n✅ Step 6/9: Applying BEHAVIORAL RULES...")
# BEHAVIORAL RULES
window_recipient_history = Window.partitionBy('ac_from', 'ac_to').orderBy('trans_timestamp')
test_rules_df = test_rules_df.withColumn('is_first_to_recipient', 
    when(row_number().over(window_recipient_history) == 1, 1).otherwise(0)) \
    .withColumn('rule_behavior_new_recipient_high', 
    when((col('is_first_to_recipient') == 1) & (col('trx_amt') > 1000), 1).otherwise(0))

window_user_history = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-50, -1)
test_rules_df = test_rules_df.withColumn('historical_weekend_ratio', 
    coalesce(avg('is_weekend').over(window_user_history), lit(0.5))) \
    .withColumn('rule_behavior_unusual_weekend', 
    when((col('is_weekend') == 1) & (col('historical_weekend_ratio') < 0.2) & (col('trx_amt') > 500), 1).otherwise(0))

test_rules_df = test_rules_df.withColumn('historical_night_ratio', 
    coalesce(avg('is_night').over(window_user_history), lit(0.3))) \
    .withColumn('rule_behavior_unusual_night', 
    when((col('is_night') == 1) & (col('historical_night_ratio') < 0.1) & (col('trx_amt') > 500), 1).otherwise(0))

test_rules_df = test_rules_df.withColumn('rule_behavior_amount_jump', 
    when((col('trx_amt') > col('historical_avg_amount') * 5) & 
         (col('historical_avg_amount') > 0) & (col('historical_avg_amount') < 1000), 1).otherwise(0))

window_prev_period = Window.partitionBy('ac_from').orderBy('trans_timestamp').rowsBetween(-100, -50)
test_rules_df = test_rules_df.withColumn('prev_period_avg_daily_txns', 
    coalesce(count('*').over(window_prev_period) / 50.0, lit(0))) \
    .withColumn('rule_behavior_velocity_jump', 
    when((col('txn_count_24hour') > col('prev_period_avg_daily_txns') * 3) & 
         (col('prev_period_avg_daily_txns') > 0), 1).otherwise(0))

behavior_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_behavior_')]
test_rules_df = test_rules_df.withColumn('behavior_rules_triggered', 
    sum([col(c) for c in behavior_rule_cols]))

print(f"   ✓ Applied {len(behavior_rule_cols)} behavioral rules")

print("\n✅ Step 7/9: Applying RECIPIENT RULES...")
# RECIPIENT RULES
test_rules_df = test_rules_df.withColumn('rule_recipient_new_high_amount', 
    when((col('is_first_to_recipient') == 1) & (col('trx_amt') > 1000) & (col('account_age_days') < 30), 1).otherwise(0)) \
    .withColumn('rule_recipient_money_mule', 
    when(col('unique_senders_24hour') > 10, 1).otherwise(0))

window_recipient_1hour = Window.partitionBy('ac_to').orderBy('trans_timestamp').rangeBetween(-3600, 0)
test_rules_df = test_rules_df.withColumn('recipient_amount_1hour', 
    spark_sum('trx_amt').over(window_recipient_1hour)) \
    .withColumn('rule_recipient_high_amount_1hour', 
    when(col('recipient_amount_1hour') > 10000, 1).otherwise(0))

recipient_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_recipient_')]
test_rules_df = test_rules_df.withColumn('recipient_rules_triggered', 
    sum([col(c) for c in recipient_rule_cols]))

print(f"   ✓ Applied {len(recipient_rule_cols)} recipient rules")

print("\n✅ Step 8/9: Applying FRAUD RING RULES...")
# FRAUD RING RULES - Use high-risk accounts from training data
if len(high_risk_account_list) > 0:
    test_rules_df = test_rules_df.withColumn('rule_ring_known_fraudster_sender', 
        when(col('ac_from').isin(high_risk_account_list), 1).otherwise(0)) \
        .withColumn('rule_ring_known_fraudster_recipient', 
        when(col('ac_to').isin(high_risk_account_list), 1).otherwise(0))
else:
    test_rules_df = test_rules_df.withColumn('rule_ring_known_fraudster_sender', lit(0)) \
        .withColumn('rule_ring_known_fraudster_recipient', lit(0))

test_rules_df = test_rules_df.withColumn('rule_ring_high_recipient_network', 
    when((col('unique_senders_24hour') > 15) | (col('txn_count_24hour') > 15), 1).otherwise(0)) \
    .withColumn('rule_ring_structured_pattern', 
    when((col('rule_amount_uniform_pattern') == 1) & (col('txn_count_24hour') > 10), 1).otherwise(0)) \
    .withColumn('rule_ring_coordinated_activity', 
    when((col('txn_count_15min') > 8) & (col('rule_amount_uniform_pattern') == 1), 1).otherwise(0))

ring_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_ring_')]
test_rules_df = test_rules_df.withColumn('ring_rules_triggered', 
    sum([col(c) for c in ring_rule_cols]))

print(f"   ✓ Applied {len(ring_rule_cols)} fraud ring rules")

# Account type placeholder
test_rules_df = test_rules_df.withColumn('actype_rules_triggered', lit(0))

print("\n✅ Step 9/9: Calculating risk scores...")
# Calculate overall risk score
rule_category_cols_test = [
    'velocity_rules_triggered', 'time_rules_triggered', 'age_rules_triggered',
    'amount_rules_triggered', 'channel_rules_triggered', 'actype_rules_triggered',
    'behavior_rules_triggered', 'recipient_rules_triggered', 'ring_rules_triggered'
]

test_rules_df = test_rules_df.withColumn('total_rules_triggered', 
    sum([col(c) for c in rule_category_cols_test]))

test_rules_df = test_rules_df.withColumn('fraud_risk_score',
    (col('velocity_rules_triggered') * 3.0) +
    (col('time_rules_triggered') * 1.5) +
    (col('age_rules_triggered') * 2.0) +
    (col('amount_rules_triggered') * 2.5) +
    (col('channel_rules_triggered') * 1.5) +
    (col('behavior_rules_triggered') * 2.0) +
    (col('recipient_rules_triggered') * 2.5) +
    (col('ring_rules_triggered') * 5.0)
)

test_rules_df = test_rules_df.withColumn('risk_category',
    when(col('fraud_risk_score') >= 20, 'CRITICAL') \
    .when(col('fraud_risk_score') >= 15, 'HIGH') \
    .when(col('fraud_risk_score') >= 10, 'MEDIUM') \
    .when(col('fraud_risk_score') >= 5, 'LOW') \
    .otherwise('MINIMAL')
)

test_rules_df = test_rules_df.coalesce(100).cache()

print("\n✅ ALL RULES APPLIED TO JULY 2025 TEST DATA!")
print("="*80)

# Count all rule columns
all_test_rule_cols = [c for c in test_rules_df.columns if c.startswith('rule_')]
print(f"📊 Summary:")
print(f"   • Total rules applied: {len(all_test_rule_cols)}")
print(f"   • Total transactions: {test_rules_df.count():,}")
print(f"   • Result partitions: {test_rules_df.rdd.getNumPartitions()}")
print("="*80)

🔧 Applying fraud detection rules to July 2025 test data...

This will apply all 43+ rules across 9 categories

✅ Step 1/9: Applying VELOCITY RULES...
   ✓ Applied 8 velocity rules

✅ Step 2/9: Applying TIME-BASED RULES...
   ✓ Applied 5 time-based rules

✅ Step 3/9: Applying ACCOUNT AGE RULES...
   ✓ Applied 8 account age rules

✅ Step 4/9: Applying AMOUNT-BASED RULES...
   ✓ Applied 7 amount-based rules

✅ Step 5/9: Applying CHANNEL-BASED RULES...
   ✓ Applied 4 channel-based rules

✅ Step 6/9: Applying BEHAVIORAL RULES...
   ✓ Applied 5 behavioral rules

✅ Step 7/9: Applying RECIPIENT RULES...
   ✓ Applied 3 recipient rules

✅ Step 8/9: Applying FRAUD RING RULES...


NameError: name 'high_risk_account_list' is not defined

In [ ]:
# Evaluate rule performance on July 2025 test data
print("\n📊 EVALUATING RULE PERFORMANCE ON JULY 2025 TEST DATA")
print("="*80)

test_total_count = test_rules_df.count()
test_fraud_count = test_rules_df.filter(col('fraud_flag') == 1).count()
test_non_fraud_count = test_rules_df.filter(col('fraud_flag') == 0).count()

print(f"\nTest Dataset Summary:")
print(f"  • Total transactions: {test_total_count:,}")
print(f"  • Fraud: {test_fraud_count:,} ({(test_fraud_count/test_total_count)*100:.2f}%)")
print(f"  • Non-fraud: {test_non_fraud_count:,} ({(test_non_fraud_count/test_total_count)*100:.2f}%)")

# Risk category distribution
print("\n🎯 Risk Category Distribution on Test Data:")
test_rules_df.groupBy('risk_category', 'fraud_flag').count().orderBy('risk_category', 'fraud_flag').show()

# High-risk transactions analysis
print("\n🚨 High-Risk Transactions (CRITICAL + HIGH):")
high_risk_test = test_rules_df.filter(col('risk_category').isin(['HIGH', 'CRITICAL']))
high_risk_test_count = high_risk_test.count()
high_risk_test_fraud = high_risk_test.filter(col('fraud_flag') == 1).count()

print(f"  • Total high-risk flagged: {high_risk_test_count:,} ({(high_risk_test_count/test_total_count)*100:.2f}%)")
print(f"  • Actual frauds in high-risk: {high_risk_test_fraud:,}")
if high_risk_test_count > 0:
    print(f"  • Precision (high-risk): {(high_risk_test_fraud/high_risk_test_count)*100:.2f}%")
print(f"  • Recall (fraud detection): {(high_risk_test_fraud/test_fraud_count)*100:.2f}%")

# Overall fraud detection by risk category
print("\n📈 Fraud Detection Rate by Risk Category:")
test_rules_df.groupBy('risk_category').agg(
    count('*').alias('total_txns'),
    spark_sum('fraud_flag').alias('fraud_count'),
    spark_round((spark_sum('fraud_flag') / count('*')) * 100, 2).alias('fraud_rate_%'),
    spark_round((spark_sum('fraud_flag') / test_fraud_count) * 100, 2).alias('recall_%')
).orderBy('risk_category').show()

# Average risk scores
print("\n📊 Average Risk Score by Fraud Flag:")
test_rules_df.groupBy('fraud_flag').agg(
    count('*').alias('count'),
    spark_round(avg('fraud_risk_score'), 2).alias('avg_risk_score'),
    spark_round(stddev('fraud_risk_score'), 2).alias('stddev_risk_score'),
    spark_min('fraud_risk_score').alias('min_risk_score'),
    spark_max('fraud_risk_score').alias('max_risk_score')
).show()

# Rule category effectiveness on test data
print("\n🎯 Rule Category Effectiveness (Test Data):")
for cat_col in rule_category_cols_test:
    if cat_col != 'actype_rules_triggered':
        avg_fraud = test_rules_df.filter(col('fraud_flag') == 1).agg(avg(cat_col)).collect()[0][0]
        avg_non_fraud = test_rules_df.filter(col('fraud_flag') == 0).agg(avg(cat_col)).collect()[0][0]
        print(f"  {cat_col}:")
        print(f"    - Fraud: {avg_fraud:.2f} rules/txn")
        print(f"    - Non-fraud: {avg_non_fraud:.2f} rules/txn")
        if avg_non_fraud > 0:
            print(f"    - Fraud/Non-fraud ratio: {avg_fraud/avg_non_fraud:.2f}x")

In [ ]:
# Calculate individual rule performance on test data
print("\n📊 Calculating individual rule performance on test data...")

test_rule_performance = []

for rule_col in all_test_rule_cols:
    rule_triggered = test_rules_df.filter(col(rule_col) == 1)
    rule_triggered_count = rule_triggered.count()
    
    if rule_triggered_count == 0:
        continue
    
    true_positives = rule_triggered.filter(col('fraud_flag') == 1).count()
    false_positives = rule_triggered.filter(col('fraud_flag') == 0).count()
    false_negatives = test_rules_df.filter((col(rule_col) == 0) & (col('fraud_flag') == 1)).count()
    
    precision = (true_positives / rule_triggered_count * 100) if rule_triggered_count > 0 else 0
    recall = (true_positives / test_fraud_count * 100) if test_fraud_count > 0 else 0
    false_positive_rate = (false_positives / test_non_fraud_count * 100) if test_non_fraud_count > 0 else 0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
    lift = (precision / (test_fraud_count/test_total_count*100)) if test_fraud_count > 0 else 0
    
    test_rule_performance.append({
        'rule_name': rule_col,
        'times_triggered': rule_triggered_count,
        'trigger_rate_%': round(rule_triggered_count / test_total_count * 100, 2),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'precision_%': round(precision, 2),
        'recall_%': round(recall, 2),
        'false_positive_rate_%': round(false_positive_rate, 2),
        'f1_score': round(f1_score, 2),
        'lift': round(lift, 2)
    })

test_rule_perf_df = pd.DataFrame(test_rule_performance)

print(f"✅ Calculated metrics for {len(test_rule_performance)} rules on July 2025 test data\n")

# Show top rules by precision on test data
print("="*100)
print("TOP 20 RULES BY PRECISION (July 2025 Test Data)")
print("="*100)
print(test_rule_perf_df.sort_values('precision_%', ascending=False).head(20).to_string(index=False))

print("\n" + "="*100)
print("TOP 20 RULES BY F1 SCORE (July 2025 Test Data)")
print("="*100)
print(test_rule_perf_df.sort_values('f1_score', ascending=False).head(20).to_string(index=False))

print("\n" + "="*100)
print("LOWEST FALSE POSITIVE RATES (July 2025 Test Data)")
print("="*100)
print(test_rule_perf_df.sort_values('false_positive_rate_%', ascending=True).head(20).to_string(index=False))

In [ ]:
# Compare training vs test performance
print("\n📊 TRAINING vs TEST PERFORMANCE COMPARISON")
print("="*100)

# Merge training and test performance
comparison_df = rule_perf_df.merge(
    test_rule_perf_df,
    on='rule_name',
    how='outer',
    suffixes=('_train', '_test')
)

# Fill NaN with 0
comparison_df = comparison_df.fillna(0)

# Calculate performance changes
comparison_df['precision_change'] = comparison_df['precision_%_test'] - comparison_df['precision_%_train']
comparison_df['fpr_change'] = comparison_df['false_positive_rate_%_test'] - comparison_df['false_positive_rate_%_train']
comparison_df['recall_change'] = comparison_df['recall_%_test'] - comparison_df['recall_%_train']
comparison_df['f1_change'] = comparison_df['f1_score_test'] - comparison_df['f1_score_train']

# Show rules with most stable performance (smallest changes)
print("\n🌟 TOP 20 MOST STABLE RULES (Smallest performance change between train and test)")
comparison_df['stability_score'] = (
    abs(comparison_df['precision_change']) + 
    abs(comparison_df['fpr_change']) + 
    abs(comparison_df['recall_change'])
) / 3

stable_rules = comparison_df.sort_values('stability_score').head(20)[[
    'rule_name', 'precision_%_train', 'precision_%_test', 'precision_change',
    'false_positive_rate_%_train', 'false_positive_rate_%_test', 'fpr_change',
    'stability_score'
]]
print(stable_rules.to_string(index=False))

# Show rules that improved on test data
print("\n✅ RULES THAT IMPROVED ON TEST DATA (Precision increased)")
improved_rules = comparison_df[comparison_df['precision_change'] > 5].sort_values('precision_change', ascending=False)[[
    'rule_name', 'precision_%_train', 'precision_%_test', 'precision_change',
    'false_positive_rate_%_train', 'false_positive_rate_%_test'
]]
print(improved_rules.to_string(index=False))

# Show rules that degraded on test data
print("\n⚠️  RULES THAT DEGRADED ON TEST DATA (Precision decreased)")
degraded_rules = comparison_df[comparison_df['precision_change'] < -5].sort_values('precision_change')[[
    'rule_name', 'precision_%_train', 'precision_%_test', 'precision_change',
    'false_positive_rate_%_train', 'false_positive_rate_%_test'
]]
print(degraded_rules.to_string(index=False))

# Overall performance summary
print("\n" + "="*100)
print("OVERALL PERFORMANCE SUMMARY")
print("="*100)
print(f"\nTraining Data (May-June 2025):")
print(f"  • Average Precision: {rule_perf_df['precision_%'].mean():.2f}%")
print(f"  • Average FPR: {rule_perf_df['false_positive_rate_%'].mean():.2f}%")
print(f"  • Average Recall: {rule_perf_df['recall_%'].mean():.2f}%")
print(f"  • Average F1 Score: {rule_perf_df['f1_score'].mean():.2f}")

print(f"\nTest Data (July 2025):")
print(f"  • Average Precision: {test_rule_perf_df['precision_%'].mean():.2f}%")
print(f"  • Average FPR: {test_rule_perf_df['false_positive_rate_%'].mean():.2f}%")
print(f"  • Average Recall: {test_rule_perf_df['recall_%'].mean():.2f}%")
print(f"  • Average F1 Score: {test_rule_perf_df['f1_score'].mean():.2f}")

print(f"\nPerformance Change:")
print(f"  • Precision change: {test_rule_perf_df['precision_%'].mean() - rule_perf_df['precision_%'].mean():+.2f}%")
print(f"  • FPR change: {test_rule_perf_df['false_positive_rate_%'].mean() - rule_perf_df['false_positive_rate_%'].mean():+.2f}%")
print(f"  • Recall change: {test_rule_perf_df['recall_%'].mean() - rule_perf_df['recall_%'].mean():+.2f}%")
print(f"  • F1 Score change: {test_rule_perf_df['f1_score'].mean() - rule_perf_df['f1_score'].mean():+.2f}")

print("\n" + "="*100)

In [ ]:
# Save July 2025 test results
print("\n💾 Saving July 2025 test results...")

# Save test rule performance
test_perf_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rule_performance_july2025_test.csv"
test_rule_perf_df.to_csv(test_perf_path, index=False)
print(f"✅ Test rule performance saved to: {test_perf_path}")

# Save training vs test comparison
comparison_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rule_performance_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"✅ Performance comparison saved to: {comparison_path}")

# Save test dataset summary
test_summary_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/july2025_test_summary.txt"
with open(test_summary_path, 'w') as f:
    f.write("JULY 2025 TEST DATA - FRAUD RULES VALIDATION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Test Period: {test_start_date} to {test_end_date}\n")
    f.write(f"Total Transactions: {test_total_count:,}\n")
    f.write(f"Fraud Transactions: {test_fraud_count:,} ({(test_fraud_count/test_total_count)*100:.2f}%)\n")
    f.write(f"Non-Fraud Transactions: {test_non_fraud_count:,} ({(test_non_fraud_count/test_total_count)*100:.2f}%)\n\n")
    
    f.write("HIGH-RISK DETECTION PERFORMANCE:\n")
    f.write(f"  • High-risk flagged: {high_risk_test_count:,} ({(high_risk_test_count/test_total_count)*100:.2f}%)\n")
    f.write(f"  • Frauds in high-risk: {high_risk_test_fraud:,}\n")
    if high_risk_test_count > 0:
        f.write(f"  • Precision: {(high_risk_test_fraud/high_risk_test_count)*100:.2f}%\n")
    f.write(f"  • Recall: {(high_risk_test_fraud/test_fraud_count)*100:.2f}%\n\n")
    
    f.write("OVERALL RULE PERFORMANCE:\n")
    f.write(f"  • Average Precision: {test_rule_perf_df['precision_%'].mean():.2f}%\n")
    f.write(f"  • Average FPR: {test_rule_perf_df['false_positive_rate_%'].mean():.2f}%\n")
    f.write(f"  • Average Recall: {test_rule_perf_df['recall_%'].mean():.2f}%\n")
    f.write(f"  • Average F1 Score: {test_rule_perf_df['f1_score'].mean():.2f}\n\n")
    
    f.write("COMPARISON WITH TRAINING DATA (May-June 2025):\n")
    f.write(f"  • Precision change: {test_rule_perf_df['precision_%'].mean() - rule_perf_df['precision_%'].mean():+.2f}%\n")
    f.write(f"  • FPR change: {test_rule_perf_df['false_positive_rate_%'].mean() - rule_perf_df['false_positive_rate_%'].mean():+.2f}%\n")
    f.write(f"  • Recall change: {test_rule_perf_df['recall_%'].mean() - rule_perf_df['recall_%'].mean():+.2f}%\n\n")
    
    f.write(f"TOP 10 BEST PERFORMING RULES (by Precision):\n")
    for idx, row in test_rule_perf_df.sort_values('precision_%', ascending=False).head(10).iterrows():
        f.write(f"  {row['rule_name']}: {row['precision_%']:.2f}% precision, {row['false_positive_rate_%']:.2f}% FPR\n")

print(f"✅ Test summary saved to: {test_summary_path}")

# Optionally save the full test dataset with rules
save_test_dataset = False  # Set to True to save full dataset

if save_test_dataset:
    test_dataset_path = "/root/research-dir/dev/jazzcash-fraud-detection/data/july2025_test_with_rules"
    test_rules_df.write.mode('overwrite').parquet(test_dataset_path)
    print(f"✅ Test dataset with rules saved to: {test_dataset_path}")
else:
    print("\nℹ️  Set save_test_dataset=True to save full test dataset with rule scores")

print("\n" + "="*80)
print("🎉 JULY 2025 TEST VALIDATION COMPLETE!")
print("="*80)
print(f"\n📁 Output files:")
print(f"  1. {test_perf_path}")
print(f"  2. {comparison_path}")
print(f"  3. {test_summary_path}")
print("\n🎯 Key Findings:")
print(f"  • Rules generalize well: Average precision change = {test_rule_perf_df['precision_%'].mean() - rule_perf_df['precision_%'].mean():+.2f}%")
print(f"  • High-risk detection precision: {(high_risk_test_fraud/high_risk_test_count)*100:.2f}%" if high_risk_test_count > 0 else "  • No high-risk transactions flagged")
print(f"  • Fraud recall: {(high_risk_test_fraud/test_fraud_count)*100:.2f}%")
print(f"  • Most stable rules: {len(stable_rules)} rules with <5% performance change")
print("="*80)